In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install rtdl-num-embeddings -q  # Yandex's tabular library
!pip install torch -q

import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")

In [ ]:
import pickle
import numpy as np

with open("/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-b/arcadia_v2_master_features.pkl", "rb") as f:
    master = pickle.load(f)

Xtr = master["X_pca_train"]   # already PCA + scaled
Xva = master["X_pca_val"]
Xte = master["X_pca_test"]

Y      = master["Y"]
mask   = master["label_mask"]
labels = master["label_names"]
tr = master["random_split"]["train_idx"]
va = master["random_split"]["val_idx"]
te = master["random_split"]["test_idx"]

print(f"PCA features: {Xtr.shape}")

import gc; gc.collect()

In [ ]:



import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import average_precision_score, matthews_corrcoef

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class FTTransformer(nn.Module):
    """Simplified FT-Transformer for continuous tabular features"""
    def __init__(self, n_features, d_token=64, n_blocks=3, attention_n_heads=4,
                 attention_dropout=0.2, ffn_d_hidden=128, ffn_dropout=0.1, output_dim=1):
        super().__init__()
        # Feature tokenizer (continuous)
        self.feature_tokens = nn.Parameter(torch.randn(n_features, d_token) * 0.02)
        self.feature_bias = nn.Parameter(torch.zeros(n_features, d_token))
        
        # CLS token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_token) * 0.02)
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_token, nhead=attention_n_heads,
                dim_feedforward=ffn_d_hidden, dropout=ffn_dropout,
                activation="gelu", batch_first=True, norm_first=True
            )
            for _ in range(n_blocks)
        ])
        
        # Output head
        self.norm = nn.LayerNorm(d_token)
        self.head = nn.Linear(d_token, output_dim)
    
    def forward(self, x):
        # x: (B, n_features) → (B, n_features, d_token)
        x = x.unsqueeze(-1) * self.feature_tokens.unsqueeze(0) + self.feature_bias.unsqueeze(0)
        
        # Prepend CLS
        cls = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat([cls, x], dim=1)
        
        # Transformer
        for block in self.blocks:
            x = block(x)
        
        # Output from CLS
        x = self.norm(x[:, 0])
        return self.head(x).squeeze(-1)


def train_ft_transformer(ab_idx, ab_name, epochs=50, batch_size=128, lr=3e-4):
    m_tr = mask[tr, ab_idx]
    m_va = mask[va, ab_idx]
    m_te = mask[te, ab_idx]
    
    if m_tr.sum() < 50:
        return None
    
    # Prepare data
    X_train = torch.from_numpy(Xtr[m_tr]).float()
    y_train = torch.from_numpy(Y[tr, ab_idx][m_tr].astype(np.float32))
    X_val = torch.from_numpy(Xva[m_va]).float().to(DEVICE)
    y_val = Y[va, ab_idx][m_va]
    X_test = torch.from_numpy(Xte[m_te]).float().to(DEVICE)
    y_test = Y[te, ab_idx][m_te]
    
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    
    model = FTTransformer(n_features=Xtr.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    pos_weight = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    best_val_pr = 0
    best_test_pr = 0
    best_test_mcc = 0
    
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        
        # Eval
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val).cpu().numpy()
            val_probs = 1 / (1 + np.exp(-val_logits))
            val_pr = average_precision_score(y_val, val_probs)
            
            if val_pr > best_val_pr:
                best_val_pr = val_pr
                test_logits = model(X_test).cpu().numpy()
                test_probs = 1 / (1 + np.exp(-test_logits))
                best_test_pr = average_precision_score(y_test, test_probs)
                best_test_mcc = matthews_corrcoef(y_test, (test_probs > 0.5).astype(int))
    
    return best_test_pr, best_test_mcc


# Run for all antibiotics
print("Training FT-Transformer per antibiotic...")
results = {}
import time
for i, ab in enumerate(labels):
    t0 = time.time()
    res = train_ft_transformer(i, ab, epochs=40)
    if res:
        pr, mcc = res
        results[ab] = {"PR": pr, "MCC": mcc}
        print(f"  {ab}: PR={pr:.4f} MCC={mcc:.4f} ({time.time()-t0:.0f}s)")

# Summary
print("\n" + "="*60)
print("FT-TRANSFORMER RESULTS")
print("="*60)
prs = [results[ab]["PR"] for ab in labels if ab in results]
mccs = [results[ab]["MCC"] for ab in labels if ab in results]
print(f"AVG PR-AUC: {np.mean(prs):.4f}")
print(f"AVG MCC:    {np.mean(mccs):.4f}")

In [ ]:
"""
PHASE C1 TEST RUN: MLP only (verify pipeline works)
"""

import os, sys, pickle, time, warnings, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    average_precision_score, matthews_corrcoef, f1_score, fbeta_score,
    roc_curve, brier_score_loss
)
warnings.filterwarnings("ignore")

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
"""
LOAD MASTER FEATURES
"""

# UPDATE PATH if your Phase B commit slug is different
MASTER_PKL = "/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-b/arcadia_v2_master_features.pkl"

with open(MASTER_PKL, "rb") as f:
    master = pickle.load(f)

Xtr = master["X_pca_train"].astype(np.float32)
Xva = master["X_pca_val"].astype(np.float32)
Xte = master["X_pca_test"].astype(np.float32)
Y = master["Y"].astype(np.float32)
mask = master["label_mask"]
labels = list(master["label_names"])
rs = master["random_split"]
idx_tr, idx_va, idx_te = rs["train_idx"], rs["val_idx"], rs["test_idx"]

# Sanity
assert Xtr.shape[0] == len(idx_tr)
assert not np.isnan(Xtr).any()
print(f"✅ Loaded: PCA {Xtr.shape[1]}d, Train/Val/Test = {Xtr.shape[0]}/{Xva.shape[0]}/{Xte.shape[0]}")
print(f"   Antibiotics: {labels}")

INPUT_DIM = Xtr.shape[1]

In [ ]:
"""
MLP MODEL + TRAINING
"""

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 256, 128], dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.GELU())
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)


def evaluate_predictions(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob > threshold).astype(int)
    metrics = {
        "pr_auc": average_precision_score(y_true, y_prob),
        "mcc": matthews_corrcoef(y_true, y_pred) if len(np.unique(y_pred)) > 1 else 0.0,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "brier": brier_score_loss(y_true, y_prob),
    }
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    spec = 1 - fpr
    valid = spec >= 0.90
    metrics["sens_at_90spec"] = tpr[valid].max() if valid.any() else 0.0
    return metrics


def find_best_threshold(y_true, y_prob, metric="mcc"):
    best_thr, best_score = 0.5, -float("inf")
    for thr in np.arange(0.1, 0.9, 0.02):
        y_pred = (y_prob > thr).astype(int)
        if len(np.unique(y_pred)) < 2:
            continue
        if metric == "mcc":
            s = matthews_corrcoef(y_true, y_pred)
        else:
            s = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
        if s > best_score:
            best_score = s
            best_thr = thr
    return best_thr


def train_mlp(Xtr, ytr, Xva, yva, Xte, yte, epochs=80, batch_size=128, lr=3e-4):
    Xtr_t = torch.from_numpy(Xtr).float()
    ytr_t = torch.from_numpy(ytr).float()
    Xva_t = torch.from_numpy(Xva).float().to(DEVICE)
    Xte_t = torch.from_numpy(Xte).float().to(DEVICE)
    
    train_loader = DataLoader(
        TensorDataset(Xtr_t, ytr_t),
        batch_size=batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(SEED)
    )
    
    model = MLP(Xtr.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    
    n_pos = ytr.sum()
    n_neg = len(ytr) - n_pos
    pw = torch.tensor([n_neg / max(n_pos, 1)]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    
    best_val_pr, best_test_probs, best_val_probs, best_epoch = -1, None, None, -1
    patience_count, patience = 0, 15
    
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        
        model.eval()
        with torch.no_grad():
            val_logits = model(Xva_t).cpu().numpy()
            val_probs = 1 / (1 + np.exp(-val_logits))
            val_pr = average_precision_score(yva, val_probs)
        
        if val_pr > best_val_pr:
            best_val_pr = val_pr
            best_epoch = epoch + 1
            with torch.no_grad():
                test_logits = model(Xte_t).cpu().numpy()
                best_test_probs = 1 / (1 + np.exp(-test_logits))
                best_val_probs = val_probs
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                break
    
    return best_val_pr, best_val_probs, best_test_probs, best_epoch


# ============================================================================
# RUN MLP PER ANTIBIOTIC
# ============================================================================
print("\n" + "="*70)
print("MLP — Per Antibiotic")
print("="*70)

results = []
for i, ab in enumerate(labels):
    m_tr = mask[idx_tr, i]
    m_va = mask[idx_va, i]
    m_te = mask[idx_te, i]
    
    if m_tr.sum() < 50:
        print(f"  {ab}: SKIP")
        continue
    
    Xtrx = Xtr[m_tr]
    ytrx = Y[idx_tr, i][m_tr]
    Xvax = Xva[m_va]
    yvax = Y[idx_va, i][m_va]
    Xtex = Xte[m_te]
    ytex = Y[idx_te, i][m_te]
    
    torch.manual_seed(SEED + i)
    t0 = time.time()
    best_val_pr, val_probs, test_probs, best_epoch = train_mlp(
        Xtrx, ytrx, Xvax, yvax, Xtex, ytex,
        epochs=80, batch_size=128, lr=3e-4
    )
    elapsed = time.time() - t0
    
    best_thr = find_best_threshold(yvax, val_probs, metric="mcc")
    test_metrics = evaluate_predictions(ytex, test_probs, threshold=best_thr)
    
    results.append({
        "model": "MLP",
        "antibiotic": ab,
        "best_epoch": best_epoch,
        "best_val_pr_auc": best_val_pr,
        "best_threshold": best_thr,
        "time_sec": elapsed,
        **{f"test_{k}": v for k, v in test_metrics.items()},
    })
    
    print(f"  {ab}: PR={test_metrics['pr_auc']:.4f} MCC={test_metrics['mcc']:.4f} "
          f"F2={test_metrics['f2']:.4f} Sens@90={test_metrics['sens_at_90spec']:.4f} "
          f"({elapsed:.0f}s, ep={best_epoch})")
    
    torch.cuda.empty_cache()
    gc.collect()

df = pd.DataFrame(results)
print(f"\n{'='*70}")
print(f"MLP RESULTS (TEST SET)")
print(f"{'='*70}")
print(f"  AVG PR-AUC: {df['test_pr_auc'].mean():.4f}")
print(f"  AVG MCC:    {df['test_mcc'].mean():.4f}")
print(f"  AVG F2:     {df['test_f2'].mean():.4f}")
print(f"  Total time: {df['time_sec'].sum():.0f}s")

df.to_csv("/kaggle/working/mlp_test_run.csv", index=False)
print("\n✅ Saved: /kaggle/working/mlp_test_run.csv")

In [ ]:
"""
PHASE C1: TABULAR NEURAL NETWORK COMPARISON
6 architectures × 7 antibiotics on PCA-1024 features

Models:
1. MLP (baseline)
2. ResNet-MLP (residual blocks)
3. TabNet (Arik & Pfister 2021)
4. SAINT (Somepalli et al. 2021)
5. TabTransformer (Huang et al. 2020)
6. AutoInt (Song et al. 2019)

All evaluated on test set with: PR-AUC, MCC, F1, F2, Sens@90Spec, Brier, ECE
"""

import os
import sys
import pickle
import json
import time
import warnings
import gc
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    average_precision_score, matthews_corrcoef, f1_score, fbeta_score,
    roc_curve, brier_score_loss, log_loss
)

warnings.filterwarnings("ignore")

# ============================================================================
# CONFIG
# ============================================================================
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Seed: {SEED}")

In [ ]:
# Tabular libraries
!pip install pytorch-tabnet -q
!pip install einops -q
print("✅ Dependencies installed")

In [ ]:
"""
LOAD MASTER FEATURES (PCA-1024)
"""

# Path to your master pkl — UPDATE this to your actual commit slug
MASTER_PKL = "/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-b/arcadia_v2_master_features.pkl"
# If using different name/path, adjust above

with open(MASTER_PKL, "rb") as f:
    master = pickle.load(f)

# Features
Xtr = master["X_pca_train"].astype(np.float32)
Xva = master["X_pca_val"].astype(np.float32)
Xte = master["X_pca_test"].astype(np.float32)

# Labels & masks
Y = master["Y"].astype(np.float32)
mask = master["label_mask"]
labels = list(master["label_names"])

# Splits
rs = master["random_split"]
idx_tr, idx_va, idx_te = rs["train_idx"], rs["val_idx"], rs["test_idx"]

# Sanity checks
assert Xtr.shape[0] == len(idx_tr), f"Train shape mismatch: {Xtr.shape[0]} vs {len(idx_tr)}"
assert Xva.shape[0] == len(idx_va)
assert Xte.shape[0] == len(idx_te)
assert Xtr.shape[1] == Xva.shape[1] == Xte.shape[1], "Feature dim mismatch"
assert not np.isnan(Xtr).any(), "NaN in train features"
assert not np.isnan(Xva).any(), "NaN in val features"
assert not np.isnan(Xte).any(), "NaN in test features"

print("="*70)
print("MASTER FEATURES LOADED")
print("="*70)
print(f"  PCA features: {Xtr.shape[1]} dims")
print(f"  Train/Val/Test: {Xtr.shape[0]}/{Xva.shape[0]}/{Xte.shape[0]}")
print(f"  Antibiotics: {labels}")
print(f"  Total samples: {len(Y)}")

# Per-antibiotic label coverage
print("\nLabel coverage:")
for i, ab in enumerate(labels):
    tr_pos = mask[idx_tr, i].sum() and (Y[idx_tr, i] * mask[idx_tr, i]).sum()
    tr_neg = mask[idx_tr, i].sum() - tr_pos
    te_pos = (Y[idx_te, i] * mask[idx_te, i]).sum()
    te_neg = mask[idx_te, i].sum() - te_pos
    print(f"  {ab}: train pos/neg = {int(tr_pos)}/{int(tr_neg)}, "
          f"test pos/neg = {int(te_pos)}/{int(te_neg)}")

In [ ]:
"""
MODEL DEFINITIONS
All return logits (no sigmoid). BCEWithLogitsLoss handles sigmoid internally.
"""

# ============================================================================
# 1. MLP (Baseline)
# ============================================================================
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 256, 128], dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.GELU())
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x).squeeze(-1)


# ============================================================================
# 2. ResNet-MLP
# ============================================================================
class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.norm = nn.BatchNorm1d(dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        h = F.gelu(self.norm(self.fc1(x)))
        h = self.dropout(h)
        h = self.fc2(h)
        return F.gelu(x + h)

class ResMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, n_blocks=4, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.ModuleList([ResBlock(hidden_dim, dropout) for _ in range(n_blocks)])
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        h = F.gelu(self.input_proj(x))
        for block in self.blocks:
            h = block(h)
        return self.head(h).squeeze(-1)


# ============================================================================
# 3. TabNet (uses pytorch_tabnet library — wraps differently)
# ============================================================================
# Handled separately in training loop (not nn.Module subclass needed for sklearn-style API)


# ============================================================================
# 4. SAINT (Self-Attention + Intersample Attention)
# Simplified version — full SAINT has both row + column attention
# ============================================================================
class SAINTBlock(nn.Module):
    def __init__(self, d_token, n_heads, dropout=0.1):
        super().__init__()
        self.col_attn = nn.MultiheadAttention(d_token, n_heads, dropout=dropout, batch_first=True)
        self.col_norm = nn.LayerNorm(d_token)
        self.col_ffn = nn.Sequential(
            nn.Linear(d_token, d_token * 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_token * 2, d_token)
        )
        self.col_ffn_norm = nn.LayerNorm(d_token)
    
    def forward(self, x):
        # Column attention (over features)
        attn_out, _ = self.col_attn(x, x, x)
        x = self.col_norm(x + attn_out)
        ffn_out = self.col_ffn(x)
        x = self.col_ffn_norm(x + ffn_out)
        return x

class SAINT(nn.Module):
    def __init__(self, input_dim, d_token=32, n_heads=4, n_blocks=3, dropout=0.1):
        super().__init__()
        self.d_token = d_token
        # Continuous feature embedding (each feature becomes a token)
        self.feature_emb = nn.Parameter(torch.randn(input_dim, d_token) * 0.02)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_token) * 0.02)
        self.blocks = nn.ModuleList([SAINTBlock(d_token, n_heads, dropout) for _ in range(n_blocks)])
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, 1)
        )
    
    def forward(self, x):
        # x: (B, n_feat) → (B, n_feat, d_token)
        x = x.unsqueeze(-1) * self.feature_emb.unsqueeze(0)
        # Prepend CLS
        cls = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat([cls, x], dim=1)
        for block in self.blocks:
            x = block(x)
        return self.head(x[:, 0]).squeeze(-1)


# ============================================================================
# 5. TabTransformer (simplified for continuous features)
# ============================================================================
class TabTransformer(nn.Module):
    def __init__(self, input_dim, d_token=32, n_heads=4, n_layers=4, dropout=0.1):
        super().__init__()
        self.feature_emb = nn.Parameter(torch.randn(input_dim, d_token) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, dim_feedforward=d_token*4,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_token * input_dim),
            nn.Linear(d_token * input_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )
    
    def forward(self, x):
        x = x.unsqueeze(-1) * self.feature_emb.unsqueeze(0)
        x = self.transformer(x)
        x = x.flatten(1)
        return self.head(x).squeeze(-1)


# ============================================================================
# 6. AutoInt (Multi-head self-attention for feature interactions)
# ============================================================================
class AutoInt(nn.Module):
    def __init__(self, input_dim, d_token=16, n_heads=4, n_layers=3, dropout=0.1):
        super().__init__()
        self.d_token = d_token
        self.feature_emb = nn.Parameter(torch.randn(input_dim, d_token) * 0.02)
        self.attn_layers = nn.ModuleList([
            nn.MultiheadAttention(d_token, n_heads, dropout=dropout, batch_first=True)
            for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(d_token) for _ in range(n_layers)])
        self.head = nn.Sequential(
            nn.Linear(d_token * input_dim, 1)
        )
    
    def forward(self, x):
        x = x.unsqueeze(-1) * self.feature_emb.unsqueeze(0)
        for attn, norm in zip(self.attn_layers, self.norms):
            attn_out, _ = attn(x, x, x)
            x = norm(x + attn_out)
        x = x.flatten(1)
        return self.head(x).squeeze(-1)


print("✅ All model definitions ready")

In [ ]:
"""
TRAINING UTILITIES
"""

def evaluate_predictions(y_true, y_prob, threshold=0.5):
    """Return all metrics for a single antibiotic"""
    y_pred = (y_prob > threshold).astype(int)
    
    metrics = {
        "pr_auc": average_precision_score(y_true, y_prob),
        "mcc": matthews_corrcoef(y_true, y_pred) if len(np.unique(y_pred)) > 1 else 0.0,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "brier": brier_score_loss(y_true, y_prob),
    }
    
    # Sensitivity at 90% specificity
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    spec = 1 - fpr
    valid = spec >= 0.90
    metrics["sens_at_90spec"] = tpr[valid].max() if valid.any() else 0.0
    
    return metrics


def find_best_threshold(y_true, y_prob, metric="mcc"):
    """Find threshold maximizing chosen metric (on val set)"""
    best_thr = 0.5
    best_score = -float("inf")
    for thr in np.arange(0.1, 0.9, 0.02):
        y_pred = (y_prob > thr).astype(int)
        if len(np.unique(y_pred)) < 2:
            continue
        if metric == "mcc":
            s = matthews_corrcoef(y_true, y_pred)
        elif metric == "f2":
            s = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
        else:
            s = f1_score(y_true, y_pred, zero_division=0)
        if s > best_score:
            best_score = s
            best_thr = thr
    return best_thr


def train_torch_model(model, Xtr, ytr, Xva, yva, Xte, yte,
                      epochs=80, batch_size=128, lr=3e-4, weight_decay=1e-5,
                      pos_weight=None, patience=15, verbose=False):
    """
    Train a PyTorch model. Test set NEVER touched during training.
    Returns: best_val_pr_auc, test_predictions, best_epoch
    """
    Xtr_t = torch.from_numpy(Xtr).float()
    ytr_t = torch.from_numpy(ytr).float()
    Xva_t = torch.from_numpy(Xva).float().to(DEVICE)
    Xte_t = torch.from_numpy(Xte).float().to(DEVICE)
    
    train_loader = DataLoader(
        TensorDataset(Xtr_t, ytr_t),
        batch_size=batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(SEED)
    )
    
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    
    if pos_weight is not None:
        pw = torch.tensor([pos_weight]).to(DEVICE)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    else:
        criterion = nn.BCEWithLogitsLoss()
    
    best_val_pr = -1.0
    best_test_probs = None
    best_val_probs = None
    best_epoch = -1
    patience_count = 0
    
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        
        # Evaluate on val (NEVER on test)
        model.eval()
        with torch.no_grad():
            val_logits = model(Xva_t).cpu().numpy()
            val_probs = 1 / (1 + np.exp(-val_logits))
            val_pr = average_precision_score(yva, val_probs)
        
        if val_pr > best_val_pr:
            best_val_pr = val_pr
            best_epoch = epoch + 1
            # Compute test predictions ONLY when val improves
            model.eval()
            with torch.no_grad():
                test_logits = model(Xte_t).cpu().numpy()
                best_test_probs = 1 / (1 + np.exp(-test_logits))
                best_val_probs = val_probs
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                if verbose:
                    print(f"      Early stop at epoch {epoch+1}")
                break
        
        if verbose and (epoch + 1) % 10 == 0:
            print(f"      Epoch {epoch+1}: val_PR={val_pr:.4f} (best={best_val_pr:.4f})")
    
    return best_val_pr, best_val_probs, best_test_probs, best_epoch


def train_and_eval_per_antibiotic(model_fn, model_name, Xtr_full, Xva_full, Xte_full,
                                   Y, mask, idx_tr, idx_va, idx_te, labels,
                                   epochs=80, batch_size=128, lr=3e-4):
    """
    Train one model architecture per antibiotic.
    Returns DataFrame with per-antibiotic metrics.
    """
    print(f"\n{'='*70}")
    print(f"MODEL: {model_name}")
    print(f"{'='*70}")
    
    results = []
    
    for i, ab in enumerate(labels):
        m_tr = mask[idx_tr, i]
        m_va = mask[idx_va, i]
        m_te = mask[idx_te, i]
        
        if m_tr.sum() < 50 or m_va.sum() < 10 or m_te.sum() < 10:
            print(f"  {ab}: SKIP (insufficient data)")
            continue
        
        Xtr = Xtr_full[m_tr]
        ytr = Y[idx_tr, i][m_tr]
        Xva = Xva_full[m_va]
        yva = Y[idx_va, i][m_va]
        Xte = Xte_full[m_te]
        yte = Y[idx_te, i][m_te]
        
        # Class imbalance pos_weight
        n_pos = ytr.sum()
        n_neg = len(ytr) - n_pos
        pos_weight = n_neg / max(n_pos, 1)
        
        # Initialize model fresh per antibiotic
        torch.manual_seed(SEED + i)  # different but reproducible
        model = model_fn(Xtr.shape[1])
        n_params = sum(p.numel() for p in model.parameters())
        
        t0 = time.time()
        try:
            best_val_pr, val_probs, test_probs, best_epoch = train_torch_model(
                model, Xtr, ytr, Xva, yva, Xte, yte,
                epochs=epochs, batch_size=batch_size, lr=lr, pos_weight=pos_weight
            )
        except Exception as e:
            print(f"  {ab}: TRAIN ERROR: {e}")
            continue
        
        elapsed = time.time() - t0
        
        # Find best threshold on VAL (not test)
        best_thr = find_best_threshold(yva, val_probs, metric="mcc")
        
        # Evaluate on TEST (final, untouched until now)
        test_metrics = evaluate_predictions(yte, test_probs, threshold=best_thr)
        
        results.append({
            "model": model_name,
            "antibiotic": ab,
            "n_params": n_params,
            "best_epoch": best_epoch,
            "best_val_pr_auc": best_val_pr,
            "best_threshold": best_thr,
            "time_sec": elapsed,
            **{f"test_{k}": v for k, v in test_metrics.items()},
        })
        
        print(f"  {ab}: PR={test_metrics['pr_auc']:.4f} MCC={test_metrics['mcc']:.4f} "
              f"F2={test_metrics['f2']:.4f} Sens@90={test_metrics['sens_at_90spec']:.4f} "
              f"({elapsed:.0f}s, ep={best_epoch})")
        
        # Cleanup
        del model
        torch.cuda.empty_cache()
        gc.collect()
    
    df = pd.DataFrame(results)
    avg_pr = df["test_pr_auc"].mean()
    avg_mcc = df["test_mcc"].mean()
    print(f"\n  AVG: PR={avg_pr:.4f} MCC={avg_mcc:.4f}")
    return df

print("✅ Training utilities ready")

In [ ]:
"""
RUN ALL 6 MODELS
Total time estimate: 2-4 hours on T4
"""

EPOCHS = 80
BATCH_SIZE = 128
LR = 3e-4

INPUT_DIM = Xtr.shape[1]  # 1024

all_results = []

# ============================================================================
# 1. MLP
# ============================================================================
def mlp_factory(input_dim):
    return MLP(input_dim, hidden_dims=[512, 256, 128], dropout=0.3)

df_mlp = train_and_eval_per_antibiotic(
    mlp_factory, "MLP", Xtr, Xva, Xte, Y, mask,
    idx_tr, idx_va, idx_te, labels,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR
)
all_results.append(df_mlp)

# ============================================================================
# 2. ResNet-MLP
# ============================================================================
def resmlp_factory(input_dim):
    return ResMLP(input_dim, hidden_dim=256, n_blocks=4, dropout=0.2)

df_resmlp = train_and_eval_per_antibiotic(
    resmlp_factory, "ResMLP", Xtr, Xva, Xte, Y, mask,
    idx_tr, idx_va, idx_te, labels,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR
)
all_results.append(df_resmlp)

# ============================================================================
# 3. TabNet (uses pytorch_tabnet library directly)
# ============================================================================
from pytorch_tabnet.tab_model import TabNetClassifier

print(f"\n{'='*70}")
print(f"MODEL: TabNet")
print(f"{'='*70}")

tabnet_results = []
for i, ab in enumerate(labels):
    m_tr = mask[idx_tr, i]
    m_va = mask[idx_va, i]
    m_te = mask[idx_te, i]
    if m_tr.sum() < 50:
        continue
    
    Xtrx = Xtr[m_tr]
    ytrx = Y[idx_tr, i][m_tr].astype(np.int64)
    Xvax = Xva[m_va]
    yvax = Y[idx_va, i][m_va].astype(np.int64)
    Xtex = Xte[m_te]
    ytex = Y[idx_te, i][m_te]
    
    t0 = time.time()
    clf = TabNetClassifier(
        n_d=24, n_a=24, n_steps=5,
        gamma=1.5, lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2),
        scheduler_params={"step_size":10, "gamma":0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type="entmax", seed=SEED, verbose=0,
        device_name=str(DEVICE),
    )
    clf.fit(
        Xtrx, ytrx,
        eval_set=[(Xvax, yvax)],
        eval_metric=["auc"],
        max_epochs=100, patience=15,
        batch_size=256, virtual_batch_size=128,
    )
    val_probs = clf.predict_proba(Xvax)[:, 1]
    test_probs = clf.predict_proba(Xtex)[:, 1]
    elapsed = time.time() - t0
    
    best_thr = find_best_threshold(yvax, val_probs, metric="mcc")
    test_metrics = evaluate_predictions(ytex, test_probs, threshold=best_thr)
    
    tabnet_results.append({
        "model": "TabNet",
        "antibiotic": ab,
        "best_val_pr_auc": average_precision_score(yvax, val_probs),
        "best_threshold": best_thr,
        "time_sec": elapsed,
        **{f"test_{k}": v for k, v in test_metrics.items()},
    })
    print(f"  {ab}: PR={test_metrics['pr_auc']:.4f} MCC={test_metrics['mcc']:.4f} ({elapsed:.0f}s)")
    
    del clf
    torch.cuda.empty_cache()
    gc.collect()

df_tabnet = pd.DataFrame(tabnet_results)
print(f"\n  AVG: PR={df_tabnet['test_pr_auc'].mean():.4f} MCC={df_tabnet['test_mcc'].mean():.4f}")
all_results.append(df_tabnet)

# ============================================================================
# 4. SAINT
# ============================================================================
def saint_factory(input_dim):
    return SAINT(input_dim, d_token=32, n_heads=4, n_blocks=3, dropout=0.1)

df_saint = train_and_eval_per_antibiotic(
    saint_factory, "SAINT", Xtr, Xva, Xte, Y, mask,
    idx_tr, idx_va, idx_te, labels,
    epochs=EPOCHS, batch_size=64, lr=2e-4  # smaller batch for memory
)
all_results.append(df_saint)

# ============================================================================
# 5. TabTransformer
# ============================================================================
def tabtrans_factory(input_dim):
    return TabTransformer(input_dim, d_token=16, n_heads=4, n_layers=3, dropout=0.1)
    # smaller d_token for memory (1024 features × 32 = 32K params per token, too much)

df_tabtrans = train_and_eval_per_antibiotic(
    tabtrans_factory, "TabTransformer", Xtr, Xva, Xte, Y, mask,
    idx_tr, idx_va, idx_te, labels,
    epochs=EPOCHS, batch_size=64, lr=2e-4
)
all_results.append(df_tabtrans)

# ============================================================================
# 6. AutoInt
# ============================================================================
def autoint_factory(input_dim):
    return AutoInt(input_dim, d_token=16, n_heads=4, n_layers=3, dropout=0.1)

df_autoint = train_and_eval_per_antibiotic(
    autoint_factory, "AutoInt", Xtr, Xva, Xte, Y, mask,
    idx_tr, idx_va, idx_te, labels,
    epochs=EPOCHS, batch_size=64, lr=2e-4
)
all_results.append(df_autoint)

# ============================================================================
# COMBINE & SAVE
# ============================================================================
combined = pd.concat(all_results, ignore_index=True)
combined.to_csv("/kaggle/working/tabular_nn_per_antibiotic.csv", index=False)
print(f"\n✅ Per-antibiotic results saved")

# Summary by model
summary = combined.groupby("model").agg({
    "test_pr_auc": "mean",
    "test_mcc": "mean",
    "test_f1": "mean",
    "test_f2": "mean",
    "test_sens_at_90spec": "mean",
    "test_brier": "mean",
    "time_sec": "sum",
}).reset_index()
summary = summary.sort_values("test_pr_auc", ascending=False)
summary.to_csv("/kaggle/working/tabular_nn_summary.csv", index=False)

print("\n" + "="*70)
print("FINAL SUMMARY (TEST SET)")
print("="*70)
print(summary.to_string(index=False))

In [ ]:
"""
PHASE C2: 1D CNN ON RAW SNP SEQUENCES (PER CONTIG)
Real sequence-aware NN architecture for AMR

Approach:
- Group 535K SNPs by contig (1271 total)
- Keep top K contigs by SNP count (covers ~80% of variance)
- Pad/truncate per contig to fixed length
- Multi-stream 1D CNN (one stream per contig)
- Concatenate features → classifier head
"""

import os, pickle, time, gc, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import (
    average_precision_score, matthews_corrcoef, f1_score, fbeta_score,
    roc_curve, brier_score_loss
)
warnings.filterwarnings("ignore")

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

In [ ]:
"""
LOAD RAW SNP MATRIX + METADATA
"""

# Phase A SNP matrix (raw 535K)
SNP_PKL = "/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/arcadia_v2_snp_matrix.pkl"

# Phase B master features (for splits + labels — already aligned)
MASTER_PKL = "/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-b/arcadia_v2_master_features.pkl"

print("Loading raw SNP matrix...")
with open(SNP_PKL, "rb") as f:
    snp_data = pickle.load(f)

X_raw = snp_data["X_snp"]  # int8, (6962, 535439)
positions = snp_data["snp_positions"]
print(f"  Raw SNP shape: {X_raw.shape}")

print("\nLoading master features (for splits + labels)...")
with open(MASTER_PKL, "rb") as f:
    master = pickle.load(f)

Y = master["Y"].astype(np.float32)
mask = master["label_mask"]
labels = list(master["label_names"])
rs = master["random_split"]
idx_tr, idx_va, idx_te = rs["train_idx"], rs["val_idx"], rs["test_idx"]

# Verify sample alignment between SNP matrix and master
master_genomes = master["genome_names"]
snp_genomes = snp_data["selected_genome_names"]
assert np.array_equal(master_genomes, snp_genomes), "Sample order mismatch!"

print(f"  Train/Val/Test: {len(idx_tr)}/{len(idx_va)}/{len(idx_te)}")
print(f"  Antibiotics: {labels}")
print(f"✅ Data aligned")

In [ ]:
"""
GROUP SNPS BY CONTIG, KEEP TOP-K
"""

# Parse positions: "CONTIG:POS:REF>ALTS"
print("Parsing SNP positions to contigs...")
contig_indices = defaultdict(list)  # contig_name → [snp_idx, ...]
for snp_idx, pos_str in enumerate(positions):
    contig_name = pos_str.split(":")[0]
    contig_indices[contig_name].append(snp_idx)

# Stats
n_contigs_total = len(contig_indices)
contig_sizes = sorted([(c, len(idx)) for c, idx in contig_indices.items()], 
                      key=lambda x: -x[1])

print(f"  Total contigs: {n_contigs_total}")
print(f"  Top 10 contigs by SNP count:")
for c, n in contig_sizes[:10]:
    print(f"    {c}: {n} SNPs")
print(f"  Bottom: {contig_sizes[-1]}")

# Strategy: Keep contigs covering top X% SNPs (memory-efficient)
SNP_COVERAGE_TARGET = 0.80  # cover 80% of total SNPs

total_snps = sum(n for _, n in contig_sizes)
cumulative = 0
top_contigs = []
for c, n in contig_sizes:
    cumulative += n
    top_contigs.append((c, n))
    if cumulative / total_snps >= SNP_COVERAGE_TARGET:
        break

n_top_contigs = len(top_contigs)
total_kept_snps = sum(n for _, n in top_contigs)

print(f"\n  Keeping top {n_top_contigs} contigs (covers {total_kept_snps}/{total_snps} = {total_kept_snps/total_snps:.1%} SNPs)")

# Get max contig length for padding
max_len = max(n for _, n in top_contigs)
print(f"  Max contig length: {max_len}")
print(f"  Memory per sample (padded, int8): {n_top_contigs * max_len / 1e6:.2f} MB")

# This may still be too large! Check
total_memory_gb = n_top_contigs * max_len * 6962 / 1e9
print(f"  Total padded matrix memory: {total_memory_gb:.2f} GB")

# If too large, reduce coverage or contig count
if total_memory_gb > 8:
    print("\n⚠ Too large. Reducing strategy: top 50 contigs only")
    top_contigs = contig_sizes[:50]
    n_top_contigs = len(top_contigs)
    max_len = max(n for _, n in top_contigs)
    total_memory_gb = n_top_contigs * max_len * 6962 / 1e9
    print(f"  Adjusted: {n_top_contigs} contigs, max_len={max_len}, total={total_memory_gb:.2f} GB")

In [ ]:
"""
BUILD PADDED CONTIG TENSORS
Memory-efficient: stay in int8, only convert to float in batch
"""

print("Building per-contig padded arrays...")

# Strategy: store as list of int8 numpy arrays, one per contig
# Each array shape: (n_samples, contig_length)
# Pad with 0 (= reference allele in our encoding)

contig_arrays = []
contig_names = []
n_samples = X_raw.shape[0]

t0 = time.time()
for c_idx, (contig_name, n_snps) in enumerate(top_contigs):
    snp_idx_list = np.array(contig_indices[contig_name], dtype=np.int32)
    contig_x = X_raw[:, snp_idx_list]  # (n_samples, n_snps_this_contig) int8
    
    # Pad to max_len (with 0 = ref allele)
    if contig_x.shape[1] < max_len:
        pad_width = max_len - contig_x.shape[1]
        contig_x = np.pad(contig_x, ((0, 0), (0, pad_width)), 
                          mode='constant', constant_values=0)
    
    contig_arrays.append(contig_x.astype(np.int8))
    contig_names.append(contig_name)
    
    if (c_idx + 1) % 10 == 0 or c_idx == len(top_contigs) - 1:
        elapsed = time.time() - t0
        print(f"  Processed {c_idx+1}/{len(top_contigs)} contigs in {elapsed:.1f}s")

# Stack into 3D: (n_samples, n_contigs, max_len)
print("\nStacking into 3D array...")
X_3d = np.stack(contig_arrays, axis=1)  # (n_samples, n_contigs, max_len)
print(f"  Shape: {X_3d.shape}")
print(f"  Memory: {X_3d.nbytes / 1e9:.2f} GB")
print(f"  Dtype: {X_3d.dtype}")

del contig_arrays, X_raw
gc.collect()
print("✅ Padded contig tensor ready")

In [ ]:
"""
DATASET CLASS — converts int8 → float on the fly per batch
"""

class SNPContigDataset(Dataset):
    def __init__(self, X_3d, y, indices, mask):
        """
        X_3d: full (n_samples, n_contigs, max_len) int8 array
        y: 1D label array for one antibiotic
        indices: sample indices to use
        mask: 1D mask for this antibiotic + indices (only labeled samples)
        """
        self.X = X_3d  # int8, kept on CPU
        self.y = y
        self.valid_indices = indices[mask]  # only labeled samples
        self.y_subset = y[self.valid_indices]
    
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, idx):
        sample_idx = self.valid_indices[idx]
        # Convert int8 → float32 only here (lazy)
        x = self.X[sample_idx].astype(np.float32)
        y = self.y_subset[idx]
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.float32)


print("✅ Dataset class ready")

In [ ]:
"""
1D CNN ARCHITECTURE
Per-contig conv → global pool → concat → classifier
"""

class PerContigCNN(nn.Module):
    """
    Multi-stream 1D CNN.
    Input: (B, n_contigs, max_len) — each contig is a sequence
    Output: logit (B,)
    """
    def __init__(self, n_contigs, max_len, 
                 conv_channels=[16, 32, 64], kernel_size=7, 
                 dropout=0.3, hidden_dim=128):
        super().__init__()
        self.n_contigs = n_contigs
        
        # Shared conv backbone (applied per contig)
        # Treat each contig as 1-channel 1D signal
        layers = []
        in_ch = 1
        for out_ch in conv_channels:
            layers.append(nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=kernel_size//2))
            layers.append(nn.BatchNorm1d(out_ch))
            layers.append(nn.GELU())
            layers.append(nn.MaxPool1d(2))
            in_ch = out_ch
        layers.append(nn.AdaptiveAvgPool1d(1))  # global average pool
        self.conv = nn.Sequential(*layers)
        
        # After conv: (B*n_contigs, conv_channels[-1], 1) → flatten → (B*n_contigs, conv_channels[-1])
        # Reshape to (B, n_contigs * conv_channels[-1])
        self.feature_dim = n_contigs * conv_channels[-1]
        
        # Classifier head
        self.head = nn.Sequential(
            nn.LayerNorm(self.feature_dim),
            nn.Linear(self.feature_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )
    
    def forward(self, x):
        # x: (B, n_contigs, max_len)
        B, NC, L = x.shape
        # Reshape to (B*NC, 1, L) for shared conv
        x = x.reshape(B * NC, 1, L)
        x = self.conv(x)  # (B*NC, C, 1)
        x = x.squeeze(-1)  # (B*NC, C)
        # Reshape back to (B, NC*C)
        x = x.reshape(B, -1)
        # Classify
        return self.head(x).squeeze(-1)


# Test instantiation
test_model = PerContigCNN(n_top_contigs, max_len)
n_params = sum(p.numel() for p in test_model.parameters())
print(f"✅ 1D CNN model — params: {n_params:,}")
print(f"  Feature dim: {test_model.feature_dim}")
del test_model

In [ ]:
"""
TRAINING UTILITIES (same protocol as MLP — fair comparison)
"""

def evaluate_predictions(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob > threshold).astype(int)
    metrics = {
        "pr_auc": average_precision_score(y_true, y_prob),
        "mcc": matthews_corrcoef(y_true, y_pred) if len(np.unique(y_pred)) > 1 else 0.0,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "brier": brier_score_loss(y_true, y_prob),
    }
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    spec = 1 - fpr
    valid = spec >= 0.90
    metrics["sens_at_90spec"] = tpr[valid].max() if valid.any() else 0.0
    return metrics


def find_best_threshold(y_true, y_prob, metric="mcc"):
    best_thr, best_score = 0.5, -float("inf")
    for thr in np.arange(0.1, 0.9, 0.02):
        y_pred = (y_prob > thr).astype(int)
        if len(np.unique(y_pred)) < 2:
            continue
        s = matthews_corrcoef(y_true, y_pred) if metric == "mcc" \
            else fbeta_score(y_true, y_pred, beta=2, zero_division=0)
        if s > best_score:
            best_score = s
            best_thr = thr
    return best_thr


def train_cnn_for_antibiotic(ab_idx, ab_name, X_3d, Y, mask,
                              idx_tr, idx_va, idx_te,
                              epochs=40, batch_size=32, lr=2e-4):
    m_tr = mask[idx_tr, ab_idx]
    m_va = mask[idx_va, ab_idx]
    m_te = mask[idx_te, ab_idx]
    
    if m_tr.sum() < 50:
        return None
    
    y_train = Y[idx_tr, ab_idx]
    y_val = Y[idx_va, ab_idx][m_va]
    y_test = Y[idx_te, ab_idx][m_te]
    
    train_ds = SNPContigDataset(X_3d, Y[:, ab_idx], idx_tr, m_tr)
    val_ds = SNPContigDataset(X_3d, Y[:, ab_idx], idx_va, m_va)
    test_ds = SNPContigDataset(X_3d, Y[:, ab_idx], idx_te, m_te)
    
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True,
        generator=torch.Generator().manual_seed(SEED + ab_idx)
    )
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    # Class imbalance
    n_pos = y_train[m_tr].sum()
    n_neg = m_tr.sum() - n_pos
    pos_weight = n_neg / max(n_pos, 1)
    
    torch.manual_seed(SEED + ab_idx)
    model = PerContigCNN(n_top_contigs, max_len).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    pw = torch.tensor([pos_weight]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    
    best_val_pr, best_val_probs, best_test_probs, best_epoch = -1, None, None, -1
    patience_count, patience = 0, 8
    
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        
        # Validate
        model.eval()
        val_probs_all = []
        with torch.no_grad():
            for xb, _ in val_loader:
                xb = xb.to(DEVICE)
                logits = model(xb).cpu().numpy()
                probs = 1 / (1 + np.exp(-logits))
                val_probs_all.append(probs)
        val_probs = np.concatenate(val_probs_all)
        val_pr = average_precision_score(y_val, val_probs)
        
        if val_pr > best_val_pr:
            best_val_pr = val_pr
            best_epoch = epoch + 1
            # Compute test only when val improves
            test_probs_all = []
            with torch.no_grad():
                for xb, _ in test_loader:
                    xb = xb.to(DEVICE)
                    logits = model(xb).cpu().numpy()
                    probs = 1 / (1 + np.exp(-logits))
                    test_probs_all.append(probs)
            best_test_probs = np.concatenate(test_probs_all)
            best_val_probs = val_probs
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                break
    
    return best_val_pr, best_val_probs, best_test_probs, best_epoch, y_val, y_test


print("✅ Training utilities ready")

In [ ]:
"""
RUN 1D CNN PER ANTIBIOTIC
"""

print("\n" + "="*70)
print("1D CNN — Per Antibiotic")
print("="*70)
print(f"Architecture: PerContigCNN (n_contigs={n_top_contigs}, max_len={max_len})")

results = []
for i, ab in enumerate(labels):
    t0 = time.time()
    
    out = train_cnn_for_antibiotic(
        i, ab, X_3d, Y, mask,
        idx_tr, idx_va, idx_te,
        epochs=40, batch_size=32, lr=2e-4
    )
    
    if out is None:
        print(f"  {ab}: SKIP")
        continue
    
    best_val_pr, val_probs, test_probs, best_epoch, y_val, y_test = out
    
    best_thr = find_best_threshold(y_val, val_probs, metric="mcc")
    test_metrics = evaluate_predictions(y_test, test_probs, threshold=best_thr)
    
    elapsed = time.time() - t0
    results.append({
        "model": "1D-CNN",
        "antibiotic": ab,
        "best_epoch": best_epoch,
        "best_val_pr_auc": best_val_pr,
        "best_threshold": best_thr,
        "time_sec": elapsed,
        **{f"test_{k}": v for k, v in test_metrics.items()},
    })
    
    print(f"  {ab}: PR={test_metrics['pr_auc']:.4f} MCC={test_metrics['mcc']:.4f} "
          f"F2={test_metrics['f2']:.4f} Sens@90={test_metrics['sens_at_90spec']:.4f} "
          f"({elapsed:.0f}s, ep={best_epoch})")
    
    torch.cuda.empty_cache()
    gc.collect()

df = pd.DataFrame(results)
print(f"\n{'='*70}")
print(f"1D CNN RESULTS (TEST SET)")
print(f"{'='*70}")
print(f"  AVG PR-AUC: {df['test_pr_auc'].mean():.4f}")
print(f"  AVG MCC:    {df['test_mcc'].mean():.4f}")
print(f"  AVG F2:     {df['test_f2'].mean():.4f}")
print(f"  Total time: {df['time_sec'].sum():.0f}s")

df.to_csv("/kaggle/working/cnn1d_results.csv", index=False)
print("\n✅ Saved: /kaggle/working/cnn1d_results.csv")

In [ ]:
"""
1D CNN ON RAW 535K SNPS — GIESSEN AUTHORS' ARCHITECTURE
Adapted for Arcadia memory constraints
"""

import os, pickle, time, gc
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.metrics import (
    average_precision_score, matthews_corrcoef, f1_score, fbeta_score,
    roc_curve, brier_score_loss
)

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# GPU memory growth
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print(f"GPU: {gpus[0].name if gpus else 'NONE'}")

# ============================================================================
# LOAD DATA
# ============================================================================
SNP_PKL = "/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/arcadia_v2_snp_matrix.pkl"
MASTER_PKL = "/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-b/arcadia_v2_master_features.pkl"

print("Loading raw SNPs...")
with open(SNP_PKL, "rb") as f:
    snp_data = pickle.load(f)
X_raw = snp_data["X_snp"]  # int8, (6962, 535439), values 0-3

# Map: 0=ref, 1=alt1, 2=alt2, 3=alt3 → matches Giessen 0-4 vocab
VOCAB_SIZE = 5  # 0,1,2,3,4 (we have 0-3, set 5 for safety)

print(f"Raw SNP: {X_raw.shape}, dtype={X_raw.dtype}")
print(f"Value range: [{X_raw.min()}, {X_raw.max()}]")

with open(MASTER_PKL, "rb") as f:
    master = pickle.load(f)

Y = master["Y"].astype(np.int32)
mask = master["label_mask"]
labels_list = list(master["label_names"])
rs = master["random_split"]
idx_tr, idx_va, idx_te = rs["train_idx"], rs["val_idx"], rs["test_idx"]

# Verify alignment
assert np.array_equal(master["genome_names"], snp_data["selected_genome_names"])
print(f"✅ Aligned. Train/Val/Test: {len(idx_tr)}/{len(idx_va)}/{len(idx_te)}")

# ============================================================================
# MODEL (Giessen architecture, scaled down for memory)
# ============================================================================
def build_giessen_cnn(maxlen, vocab_size=5, embed_dim=16, conv_dropout=0.15, dense_dropout=0.45):
    """
    Giessen authors' 6-block 1D CNN with embedding.
    Scaled embed_dim to 16 (vs Giessen 64) for memory.
    """
    inputs = keras.Input(shape=(maxlen,), dtype='int32', name="snp_input")
    
    # Embedding (key Giessen trick)
    x = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim, name="embedding")(inputs)
    
    # Block 1: aggressive pool to downsample big sequence
    x = layers.Conv1D(64, 7, padding="same", kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(4)(x)  # /4 (vs Giessen /2 — bigger downsample for our 535K)
    
    # Block 2
    x = layers.Conv1D(64, 7, padding="same", kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(4)(x)  # /16 total
    
    # Block 3
    x = layers.Conv1D(32, 5, padding="same", kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(2)(x)  # /32
    
    # Block 4
    x = layers.Conv1D(32, 5, padding="same", kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(2)(x)  # /64
    
    # Block 5
    x = layers.Conv1D(16, 3, padding="same", kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(2)(x)  # /128
    
    # Block 6
    x = layers.Conv1D(16, 3, padding="same", kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling1D()(x)
    
    # Dense head
    x = layers.Dense(64, kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dense_dropout)(x)
    
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    return keras.Model(inputs=inputs, outputs=outputs)

# Test build
model_test = build_giessen_cnn(X_raw.shape[1])
print(f"Total params: {model_test.count_params():,}")
del model_test
import keras.backend as K
K.clear_session()

# ============================================================================
# TRAINING UTILITIES
# ============================================================================
def evaluate_predictions(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob > threshold).astype(int)
    metrics = {
        "pr_auc": average_precision_score(y_true, y_prob),
        "mcc": matthews_corrcoef(y_true, y_pred) if len(np.unique(y_pred)) > 1 else 0.0,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "brier": brier_score_loss(y_true, y_prob),
    }
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    spec = 1 - fpr
    valid = spec >= 0.90
    metrics["sens_at_90spec"] = tpr[valid].max() if valid.any() else 0.0
    return metrics

def find_best_threshold(y_true, y_prob, metric="mcc"):
    best_thr, best_score = 0.5, -float("inf")
    for thr in np.arange(0.1, 0.9, 0.02):
        y_pred = (y_prob > thr).astype(int)
        if len(np.unique(y_pred)) < 2:
            continue
        s = matthews_corrcoef(y_true, y_pred) if metric == "mcc" \
            else fbeta_score(y_true, y_pred, beta=2, zero_division=0)
        if s > best_score:
            best_score = s
            best_thr = thr
    return best_thr

# ============================================================================
# TRAIN PER ANTIBIOTIC
# ============================================================================
results = []
INPUT_LEN = X_raw.shape[1]

for i, ab in enumerate(labels_list):
    print(f"\n{'='*60}")
    print(f"Training 1D CNN for {ab}")
    print(f"{'='*60}")
    
    K.clear_session()
    tf.random.set_seed(SEED + i)
    
    m_tr = mask[idx_tr, i]
    m_va = mask[idx_va, i]
    m_te = mask[idx_te, i]
    
    if m_tr.sum() < 50:
        print(f"  SKIP: insufficient data")
        continue
    
    # Get features per split (cast to int32 for embedding)
    Xtr = X_raw[idx_tr][m_tr].astype(np.int32)
    ytr = Y[idx_tr, i][m_tr].astype(np.float32)
    Xva = X_raw[idx_va][m_va].astype(np.int32)
    yva = Y[idx_va, i][m_va].astype(np.float32)
    Xte = X_raw[idx_te][m_te].astype(np.int32)
    yte = Y[idx_te, i][m_te].astype(np.float32)
    
    # Class weights
    n_neg = (ytr == 0).sum()
    n_pos = (ytr == 1).sum()
    total = n_neg + n_pos
    cw = {0: total / (2.0 * n_neg), 1: total / (2.0 * n_pos)}
    
    # Build model
    model = build_giessen_cnn(INPUT_LEN)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc")]
    )
    
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_auc", patience=25, mode="max", restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=5, min_lr=1e-6),
    ]
    
    t0 = time.time()
    history = model.fit(
        Xtr, ytr,
        batch_size=32,  # small batch for memory (535K input)
        epochs=100,
        validation_data=(Xva, yva),
        callbacks=callbacks,
        class_weight=cw,
        verbose=0,
    )
    elapsed = time.time() - t0
    
    # Predict
    val_probs = model.predict(Xva, batch_size=8, verbose=0).ravel()
    test_probs = model.predict(Xte, batch_size=8, verbose=0).ravel()
    
    # Threshold on val, eval on test
    best_thr = find_best_threshold(yva, val_probs, metric="mcc")
    test_metrics = evaluate_predictions(yte, test_probs, threshold=best_thr)
    
    n_epochs_run = len(history.history["loss"])
    results.append({
        "model": "1D-CNN-Giessen",
        "antibiotic": ab,
        "epochs_run": n_epochs_run,
        "best_threshold": best_thr,
        "time_sec": elapsed,
        **{f"test_{k}": v for k, v in test_metrics.items()},
    })
    
    print(f"  PR={test_metrics['pr_auc']:.4f} MCC={test_metrics['mcc']:.4f} "
          f"F2={test_metrics['f2']:.4f} ({elapsed:.0f}s, ep={n_epochs_run})")
    
    del model, Xtr, Xva, Xte
    K.clear_session()
    gc.collect()

df = pd.DataFrame(results)
print(f"\n{'='*60}")
print(f"GIESSEN-STYLE 1D CNN RESULTS (TEST)")
print(f"{'='*60}")
print(f"  AVG PR-AUC: {df['test_pr_auc'].mean():.4f}")
print(f"  AVG MCC:    {df['test_mcc'].mean():.4f}")
print(f"  AVG F2:     {df['test_f2'].mean():.4f}")
print(f"  Total time: {df['time_sec'].sum():.0f}s")

df.to_csv("/kaggle/working/cnn1d_giessen_arcadia.csv", index=False)
print("\n✅ Saved")

In [ ]:
import pickle
with open("/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-b/arcadia_v2_master_features.pkl", "rb") as f:
    d = pickle.load(f)
print(d.keys())

In [ ]:
"""
================================================================================
AMR-CascadeStack v6.1 — Thesis Final Tuned Edition
================================================================================

WHAT CHANGED FROM v6 (5 SAFE TUNINGS, all leak-free):

  TUNE 1 — Adaptive base-learner gating in meta-input
    v6: meta MLP always sees all 5 base probabilities, even if a base learner
        clearly underperforms (e.g. TabNet OOF=0.50 on GEN/SXT).
    v6.1: After OOF training, compute per-base-learner OOF→val gap and OOF
          PR-AUC. If a learner has gap > GATE_GAP_THR OR OOF PR-AUC < GATE_PR_THR,
          DROP that base learner from meta features for THIS antibiotic only.
          Decision uses TRAIN-side OOF + VAL — no test-set use.
          Expected: +1-2% on GEN, SXT (where TabNet was dragging).

  TUNE 2 — Train-size-aware meta MLP capacity
    v6: Fixed 64→32→16 MLP for all antibiotics (overfit on small train).
    v6.1: If train size < 3000 → smaller MLP (32→16). Otherwise keep 64→32→16.
          Standard small-data practice — reduces overfit on AMC, SXT.

  TUNE 3 — Dual-track threshold (MCC vs F2) selected on val_holdout
    v6: Only MCC-optimised threshold.
    v6.1: Compute both MCC-thr and F2-thr on val_holdout (40% of val).
          Pick the threshold that scores better on a balanced criterion
          (geometric mean of MCC and F2) on the SAME val_holdout — no test leakage.
          Expected: +1-2% on F2 (clinically valuable for AMR sensitivity).

  TUNE 4 — Snapshot averaging for meta MLP (well-established technique)
    v6: Single best meta MLP weight set selected by val_meta PR-AUC.
    v6.1: Track TOP-3 best snapshots (by val_meta PR-AUC) during training,
          average their predictions at inference. Reduces variance of meta MLP.
          Expected: +0.5-1% across antibiotics.

  TUNE 5 — Better TabNet configuration
    v6: n_d=16, n_a=16, n_steps=4, momentum=0.02, lr=2e-2 (default-ish).
    v6.1: n_d=24, n_a=24, n_steps=5, momentum=0.3, mask_type="entmax",
          higher virtual_batch_size for stability, longer patience.
          These are TabNet paper-recommended defaults for ~5k samples.
          Expected: TabNet OOF→val gap drops, fold std drops.

PRESERVED FROM v6 (do NOT change):
  ✅ 3-way OOF fold split (fit / ES / cal)
  ✅ Optuna on train-side 80/20 inner split (val never touched)
  ✅ Isotonic calibration with sigmoid fallback
  ✅ Bootstrap 95% CI for PR-AUC and MCC
  ✅ Wilcoxon signed-rank test vs best base
  ✅ Sensitivity at 90% specificity (clinical metric)
  ✅ ECE + calibration plots
  ✅ All seeds fixed (SEED=42)

WHAT THIS GIVES YOU:
  • Same leak-free guarantees as v6
  • Targeted fixes for v6's weakest 3 antibiotics (GEN, SXT, CAZ)
  • Honest, defensible tuning — every change has a published rationale
  • Expected: +1-3% PR-AUC, +1-2% MCC vs v6
  • If gains don't materialize, the diagnostic logs show why
================================================================================
"""
!pip install optuna pytorch-tabnet lightgbm -q
import pickle, random, warnings, json
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, fbeta_score, precision_score, recall_score,
    accuracy_score, precision_recall_curve,
    matthews_corrcoef, brier_score_loss, roc_curve,
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.isotonic import IsotonicRegression
from xgboost import XGBClassifier
from scipy.stats import wilcoxon

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False

try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    HAS_TABNET = True
except Exception:
    HAS_TABNET = False

try:
    import optuna
    from optuna.pruners import MedianPruner
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except Exception:
    HAS_OPTUNA = False

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ============================================================================
# CONFIG
# ============================================================================
SEED                  = 42
SPLIT_STRATEGY        = "random"
OUTPUT_DIR            = Path("/kaggle/working/cascade_v6_1_arcadia_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH          = Path("/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-b/arcadia_v2_master_features.pkl")

N_FOLDS               = 5
DEVICE                = "cuda" if torch.cuda.is_available() else "cpu"

# Optuna — train-side inner split
OPTUNA_TRIALS         = 50
OPTUNA_INNER_VAL_FRAC = 0.20

# OOF fold 3-way split
OOF_ES_FRAC           = 0.15
OOF_CAL_FRAC          = 0.15
MIN_CAL_SAMPLES       = 30

# Val split
VAL_META_FRAC         = 0.60       # 60% meta-ES / 40% threshold-holdout

# Meta MLP
META_LR               = 3e-4
META_WD               = 1e-3
META_EPOCHS           = 150
META_PAT              = 25
META_BATCH            = 128
META_SNAPSHOTS_K      = 3          # NEW: top-K snapshot averaging

# NEW: train-size-aware MLP capacity threshold
SMALL_TRAIN_THR       = 3000       # if train < this, use smaller MLP

# NEW: adaptive base gating thresholds
GATE_GAP_THR          = 0.15       # OOF→val gap > this = gate the learner
GATE_PR_THR           = 0.55       # OOF PR-AUC < this = gate the learner
MIN_BASE_AFTER_GATE   = 3          # never drop below this many base learners

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================================
# LOAD DATASET
# ============================================================================
print("=" * 80)
print("  AMR-CascadeStack v6.1 — Thesis Final Tuned Edition")
print("  5 safe tunings: gate / size-aware MLP / dual-thr / snapshots / TabNet")
print("=" * 80)
print(f"  Device  : {DEVICE}" +
      (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))
print(f"  Optuna  : {'✅' if HAS_OPTUNA else '❌'}")
print(f"  TabNet  : {'✅' if HAS_TABNET else '❌'}")
print(f"  LightGBM: {'✅' if HAS_LGB else '❌'}")

# NEW
# Load master features
with open(DATASET_PATH, "rb") as f:
    dataset = pickle.load(f)

# Master pkl has separate PCA arrays per split — reconstruct full X
X_pca_train = dataset["X_pca_train"].astype("float32")  # (4872, 1024)
X_pca_val   = dataset["X_pca_val"].astype("float32")    # (1045, 1024)
X_pca_test  = dataset["X_pca_test"].astype("float32")   # (1045, 1024)

# Get splits
rs        = dataset["random_split"]
train_idx = np.array(rs["train_idx"])
val_idx   = np.array(rs["val_idx"])
test_idx  = np.array(rs["test_idx"])

# Reconstruct full X (n_samples, n_features)
n_samples = dataset["n_samples"]  # 6962
n_features = X_pca_train.shape[1]  # 1024
X = np.zeros((n_samples, n_features), dtype="float32")
X[train_idx] = X_pca_train
X[val_idx]   = X_pca_val
X[test_idx]  = X_pca_test

Y          = dataset["Y"].astype("float32")
label_mask = dataset["label_mask"].astype(bool)
label_names = [str(n) for n in dataset["label_names"]]
ab2idx     = {name: i for i, name in enumerate(label_names)}

INPUT_DIM   = X.shape[1]
ScalerClass = StandardScaler

ACTIVE_BASE_NAMES = ["XGBoost"]
if HAS_LGB:    ACTIVE_BASE_NAMES.append("LightGBM")
ACTIVE_BASE_NAMES += ["RF", "LogReg"]
if HAS_TABNET: ACTIVE_BASE_NAMES.append("TabNet")
N_BASE_MODELS = len(ACTIVE_BASE_NAMES)

print(f"\n  X        : {X.shape}")
print(f"  Labels   : {label_names}")
print(f"  Train / Val / Test : {len(train_idx)} / {len(val_idx)} / {len(test_idx)}")
print(f"  Base models ({N_BASE_MODELS}): {' | '.join(ACTIVE_BASE_NAMES)}")
print(f"  Val split: {int(VAL_META_FRAC*100)}% meta-ES "
      f"/ {int((1-VAL_META_FRAC)*100)}% threshold-holdout")
print(f"  Fold split: 70% fit / {int(OOF_ES_FRAC*100)}% ES / "
      f"{int(OOF_CAL_FRAC*100)}% cal")
print(f"\n  TUNING SETTINGS:")
print(f"    GATE_GAP_THR={GATE_GAP_THR}  GATE_PR_THR={GATE_PR_THR}")
print(f"    SMALL_TRAIN_THR={SMALL_TRAIN_THR}  META_SNAPSHOTS_K={META_SNAPSHOTS_K}")
print()


# ============================================================================
# UTILITIES
# ============================================================================
def safe_auc(y, p):
    return float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan

def safe_pr(y, p):
    p = np.nan_to_num(np.clip(p.astype(float), 0.0, 1.0), nan=0.5)
    return float(average_precision_score(y, p)) if len(np.unique(y)) > 1 else 0.0

def best_thr_mcc(y, p):
    thresholds = np.linspace(0.05, 0.95, 181)
    best_t, best_m = 0.5, -1.0
    for t in thresholds:
        pred = (p >= t).astype(int)
        if pred.sum() == 0 or pred.sum() == len(pred):
            continue
        m = matthews_corrcoef(y, pred)
        if m > best_m:
            best_m, best_t = m, t
    return float(best_t), float(best_m)

def best_thr_f2(y, p):
    p = np.clip(p.astype(float), 1e-9, 1 - 1e-9)
    prec, rec, thr = precision_recall_curve(y, p)
    if thr.size == 0:
        return 0.5, 0.0
    f2_arr = 5 * (prec[1:] * rec[1:]) / (4 * prec[1:] + rec[1:] + 1e-12)
    best_i = int(np.nanargmax(f2_arr))
    return float(thr[best_i]), float(f2_arr[best_i])

def sensitivity_at_specificity(y_true, y_score, target_spec=0.90):
    if len(np.unique(y_true)) < 2:
        return np.nan
    fpr, tpr, _ = roc_curve(y_true, y_score)
    spec = 1.0 - fpr
    valid = spec >= target_spec
    return float(tpr[valid].max()) if valid.any() else np.nan

def compute_metrics(y, p, thr):
    p    = np.clip(p.astype(float), 0.0, 1.0)
    pred = (p >= thr).astype(int)
    sens90 = sensitivity_at_specificity(y, p, 0.90)
    return dict(
        auc    = safe_auc(y, p),
        prauc  = safe_pr(y, p),
        f1     = float(f1_score(y, pred, zero_division=0)),
        f2     = float(fbeta_score(y, pred, beta=2, zero_division=0)),
        rec    = float(recall_score(y, pred, zero_division=0)),
        prec   = float(precision_score(y, pred, zero_division=0)),
        mcc    = float(matthews_corrcoef(y, pred)),
        brier  = float(brier_score_loss(y, p)),
        acc    = float(accuracy_score(y, pred)),
        sens90 = float(sens90) if not np.isnan(sens90) else float("nan"),
        thr    = float(thr),
    )

def bootstrap_ci(y_true, y_score, metric_fn, n_boot=2000, seed=42):
    rng, scores, n = np.random.RandomState(seed), [], len(y_true)
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        try:
            scores.append(metric_fn(y_true[idx], y_score[idx]))
        except Exception:
            continue
    if len(scores) < 100:
        return (np.nan, np.nan)
    return (float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5)))

def spw(y):
    pos = float(np.sum(y == 1))
    neg = float(len(y) - pos)
    return max(1.0, neg / max(1.0, pos))

def get_data(ab):
    idx   = ab2idx[ab]
    y_all = Y[:, idx]
    m     = label_mask[:, idx]
    tr    = train_idx[m[train_idx]]
    va    = val_idx[m[val_idx]]
    te    = test_idx[m[test_idx]]
    return (X[tr], y_all[tr].astype("int32"),
            X[va], y_all[va].astype("int32"),
            X[te], y_all[te].astype("int32"))

def isotonic_calibrate(p_cal, y_cal, p_target, min_samples=MIN_CAL_SAMPLES):
    y_cal = y_cal.astype(float)
    if len(np.unique(y_cal)) < 2:
        return p_target.astype("float32")
    if len(y_cal) < min_samples:
        try:
            lo = np.log(np.clip(p_cal, 1e-7, 1-1e-7) /
                        (1 - np.clip(p_cal, 1e-7, 1-1e-7))).reshape(-1, 1)
            lr = LogisticRegression(C=1.0, solver="lbfgs", max_iter=500)
            lr.fit(lo, y_cal)
            lo_t = np.log(np.clip(p_target, 1e-7, 1-1e-7) /
                          (1 - np.clip(p_target, 1e-7, 1-1e-7))).reshape(-1, 1)
            return lr.predict_proba(lo_t)[:, 1].astype("float32")
        except Exception:
            return p_target.astype("float32")
    try:
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(p_cal, y_cal)
        return ir.predict(p_target).astype("float32")
    except Exception:
        return p_target.astype("float32")


# ============================================================================
# OPTUNA
# ============================================================================
def optuna_tune_xgb(X_inner_tr, y_inner_tr, X_inner_va, y_inner_va,
                    n_trials=OPTUNA_TRIALS):
    def objective(trial):
        params = dict(
            max_depth        = trial.suggest_int("max_depth", 3, 8),
            min_child_weight = trial.suggest_int("min_child_weight", 5, 50),
            subsample        = trial.suggest_float("subsample", 0.5, 1.0),
            colsample_bytree = trial.suggest_float("colsample_bytree", 0.3, 1.0),
            learning_rate    = trial.suggest_float("lr", 0.01, 0.15, log=True),
            gamma            = trial.suggest_float("gamma", 0.0, 5.0),
            reg_alpha        = trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            reg_lambda       = trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            n_estimators          = 3000,
            scale_pos_weight      = spw(y_inner_tr),
            eval_metric           = "aucpr",
            early_stopping_rounds = 80,
            random_state          = SEED,
            verbosity             = 0,
            tree_method           = "hist",
        )
        if DEVICE == "cuda":
            params["device"] = "cuda"
        try:
            clf = XGBClassifier(**params)
            clf.fit(X_inner_tr, y_inner_tr,
                    eval_set=[(X_inner_va, y_inner_va)], verbose=False)
            score = safe_pr(y_inner_va, clf.predict_proba(X_inner_va)[:, 1])
            trial.report(score, step=1)
            if trial.should_prune():
                raise optuna.TrialPruned()
            return score
        except optuna.TrialPruned:
            raise
        except Exception:
            return 0.0

    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=0) if HAS_OPTUNA else None
    study  = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=pruner,
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params, study.best_value


def optuna_tune_lgbm(X_inner_tr, y_inner_tr, X_inner_va, y_inner_va,
                     n_trials=OPTUNA_TRIALS):
    def objective(trial):
        params = dict(
            num_leaves        = trial.suggest_int("num_leaves", 20, 150),
            max_depth         = trial.suggest_int("max_depth", 3, 10),
            min_child_samples = trial.suggest_int("min_child_samples", 10, 60),
            subsample         = trial.suggest_float("subsample", 0.5, 1.0),
            subsample_freq    = 1,
            colsample_bytree  = trial.suggest_float("colsample_bytree", 0.3, 1.0),
            learning_rate     = trial.suggest_float("lr", 0.01, 0.15, log=True),
            reg_alpha         = trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            reg_lambda        = trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            n_estimators        = 3000,
            scale_pos_weight    = spw(y_inner_tr),
            random_state        = SEED,
            verbose             = -1,
            n_jobs              = -1,
        )
        try:
            clf = lgb.LGBMClassifier(**params)
            cbs = [lgb.early_stopping(80, verbose=False), lgb.log_evaluation(-1)]
            clf.fit(X_inner_tr, y_inner_tr,
                    eval_set=[(X_inner_va, y_inner_va)],
                    eval_metric="average_precision", callbacks=cbs)
            score = safe_pr(y_inner_va, clf.predict_proba(X_inner_va)[:, 1])
            trial.report(score, step=1)
            if trial.should_prune():
                raise optuna.TrialPruned()
            return score
        except optuna.TrialPruned:
            raise
        except Exception:
            return 0.0

    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=0) if HAS_OPTUNA else None
    study  = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=pruner,
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params, study.best_value


# ============================================================================
# BASE MODEL TRAINERS
# ============================================================================
def train_xgb_tuned(Xfit, yfit, Xes, yes, best_params):
    params = dict(
        max_depth        = best_params.get("max_depth", 4),
        min_child_weight = best_params.get("min_child_weight", 10),
        subsample        = best_params.get("subsample", 0.7),
        colsample_bytree = best_params.get("colsample_bytree", 0.7),
        learning_rate    = best_params.get("lr", 0.05),
        gamma            = best_params.get("gamma", 0.0),
        reg_alpha        = best_params.get("reg_alpha", 0.01),
        reg_lambda       = best_params.get("reg_lambda", 1.0),
        n_estimators          = 4000,
        scale_pos_weight      = spw(yfit),
        eval_metric           = "aucpr",
        early_stopping_rounds = 150,
        random_state          = SEED,
        verbosity             = 0,
        tree_method           = "hist",
    )
    if DEVICE == "cuda":
        params["device"] = "cuda"
    clf = XGBClassifier(**params)
    clf.fit(Xfit, yfit, eval_set=[(Xes, yes)], verbose=False)
    return clf


def train_lgbm_tuned(Xfit, yfit, Xes, yes, best_params):
    params = dict(
        num_leaves        = best_params.get("num_leaves", 31),
        max_depth         = best_params.get("max_depth", -1),
        min_child_samples = best_params.get("min_child_samples", 20),
        subsample         = best_params.get("subsample", 0.7),
        subsample_freq    = 1,
        colsample_bytree  = best_params.get("colsample_bytree", 0.5),
        learning_rate     = best_params.get("lr", 0.05),
        reg_alpha         = best_params.get("reg_alpha", 0.01),
        reg_lambda        = best_params.get("reg_lambda", 1.0),
        n_estimators        = 4000,
        scale_pos_weight    = spw(yfit),
        random_state        = SEED,
        verbose             = -1,
        n_jobs              = -1,
    )
    clf = lgb.LGBMClassifier(**params)
    cbs = [lgb.early_stopping(150, verbose=False), lgb.log_evaluation(-1)]
    clf.fit(Xfit, yfit, eval_set=[(Xes, yes)],
            eval_metric="average_precision", callbacks=cbs)
    return clf


def train_rf(Xfit, yfit):
    return RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=10,
        class_weight="balanced_subsample", n_jobs=-1, random_state=SEED
    ).fit(Xfit, yfit)


def train_logreg(Xfit, yfit):
    sc  = ScalerClass().fit(Xfit)
    clf = LogisticRegression(
        C=0.1, penalty="l2", solver="lbfgs",
        max_iter=3000, class_weight="balanced", random_state=SEED,
    )
    clf.fit(sc.transform(Xfit), yfit)
    return clf, sc


def train_tabnet(Xfit, yfit, Xes, yes):
    """
    TUNE 5: Better TabNet config — paper-recommended for ~5k tabular samples.
    Larger n_d/n_a, more steps, entmax mask, higher momentum, longer patience.
    """
    clf = TabNetClassifier(
        n_d                = 24,           # was 16
        n_a                = 24,           # was 16
        n_steps            = 5,            # was 4
        gamma              = 1.5,          # was 1.3
        n_independent      = 2,
        n_shared           = 2,
        momentum           = 0.3,          # was 0.02 — TabNet paper uses 0.3
        mask_type          = "entmax",     # was "sparsemax" — entmax = sharper attn
        seed               = SEED,
        verbose            = 0,
        device_name        = DEVICE,
        optimizer_fn       = torch.optim.AdamW,
        optimizer_params   = dict(lr=2e-2, weight_decay=1e-5),
        scheduler_fn       = torch.optim.lr_scheduler.CosineAnnealingLR,
        scheduler_params   = dict(T_max=80, eta_min=1e-5),
    )
    clf.fit(
        Xfit, yfit,
        eval_set=[(Xes, yes)],
        eval_metric=["auc"],
        max_epochs=200,
        patience=30,                       # was 20 — longer patience
        batch_size=512,                    # was 256 — bigger for stability
        virtual_batch_size=128,
        weights={0: 1.0, 1: spw(yfit)},
        drop_last=False,
    )
    return clf


# ============================================================================
# META FEATURES — gating-aware
# ============================================================================
def make_meta_features(probs, active_mask=None):
    """
    Build engineered meta features. If active_mask is given (boolean array
    over ACTIVE_BASE_NAMES), only retain probs of active learners.
    """
    if active_mask is not None:
        probs = probs[:, active_mask]
    probs = np.nan_to_num(np.clip(probs.astype(float), 0.0, 1.0), nan=0.5)
    n_m   = max(probs.shape[1], 1)
    feats = [probs] if probs.shape[1] > 0 else []

    if probs.shape[1] >= 1:
        p_max  = probs.max(axis=1, keepdims=True)
        p_min  = probs.min(axis=1, keepdims=True)
        p_mean = probs.mean(axis=1, keepdims=True)
        p_std  = probs.std(axis=1, keepdims=True)
        p_rng  = (p_max - p_min)
        feats += [p_max, p_min, p_mean, p_std, p_rng]

        p_clip  = np.clip(probs, 1e-8, 1 - 1e-8)
        entropy = -(p_clip * np.log(p_clip) +
                    (1 - p_clip) * np.log(1 - p_clip)).mean(axis=1, keepdims=True)
        feats.append(entropy)

        ranks = np.argsort(np.argsort(probs, axis=1), axis=1).astype(float)
        feats.append(ranks.mean(axis=1, keepdims=True) / max(n_m - 1, 1))

        votes = (probs > 0.5).sum(axis=1, keepdims=True).astype(float) / n_m
        feats.append(votes)

        mean_clipped = np.clip(p_mean, 1e-7, 1 - 1e-7)
        log_odds     = np.log(mean_clipped / (1 - mean_clipped))
        feats.append(log_odds)

        feats.append(probs.mean(axis=1, keepdims=True))   # mean again for symmetry
        feats.append(p_max * p_min)                        # interaction

    return np.hstack(feats) if feats else np.zeros((0, 0))


# ============================================================================
# META MLP — train-size-aware, with snapshot averaging
# ============================================================================
class MetaMLPSmall(nn.Module):
    """For train < SMALL_TRAIN_THR — smaller capacity to avoid overfitting."""
    def __init__(self, n_input):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_input, 32), nn.LayerNorm(32), nn.GELU(), nn.Dropout(0.30),
            nn.Linear(32, 16),                          nn.GELU(), nn.Dropout(0.15),
            nn.Linear(16, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)


class MetaMLPLarge(nn.Module):
    """For train ≥ SMALL_TRAIN_THR — original v6 capacity."""
    def __init__(self, n_input):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_input, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.25),
            nn.Linear(64, 32),      nn.LayerNorm(32), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(32, 16),                         nn.GELU(), nn.Dropout(0.10),
            nn.Linear(16, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_meta_mlp(X_meta_tr, y_tr, X_meta_val, y_val, n_train_samples):
    """
    TUNE 2 + TUNE 4: train-size-aware MLP + top-K snapshot averaging.
    Tracks top-K snapshots by val_meta PR-AUC. At inference, predictions are
    averaged across snapshots — reduces variance, classic ensemble trick.
    """
    sc    = StandardScaler().fit(X_meta_tr)
    Xtr_s = sc.transform(X_meta_tr).astype("float32")
    Xva_s = sc.transform(X_meta_val).astype("float32")

    # TUNE 2: train-size-aware capacity
    if n_train_samples < SMALL_TRAIN_THR:
        Model = MetaMLPSmall
        capacity_tag = "small"
    else:
        Model = MetaMLPLarge
        capacity_tag = "large"

    model = Model(X_meta_tr.shape[1]).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=META_LR, weight_decay=META_WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=META_EPOCHS, eta_min=1e-5)
    crit  = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([spw(y_tr)], dtype=torch.float32, device=DEVICE),
    )

    tr_dl = DataLoader(
        TensorDataset(torch.from_numpy(Xtr_s),
                      torch.from_numpy(y_tr.astype("float32"))),
        batch_size=META_BATCH, shuffle=True, drop_last=False,
    )

    # TUNE 4: track top-K snapshots
    snapshots = []   # list of (val_pr, state_dict)
    wait, last_best = 0, 0.0
    for ep in range(META_EPOCHS):
        model.train()
        for xb, yb in tr_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            crit(model(xb), yb).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            p_va = torch.sigmoid(
                model(torch.from_numpy(Xva_s).to(DEVICE))).cpu().numpy()
        pr = safe_pr(y_val, p_va)

        # Update snapshot pool
        snap_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        snapshots.append((pr, snap_state))
        snapshots.sort(key=lambda z: -z[0])    # high PR first
        snapshots = snapshots[:META_SNAPSHOTS_K]

        # Early stopping on best snapshot
        if snapshots[0][0] > last_best + 0.0005:
            last_best = snapshots[0][0]
            wait = 0
        else:
            wait += 1
            if wait >= META_PAT:
                break

    return [s for _, s in snapshots], sc, last_best, capacity_tag, Model, X_meta_tr.shape[1]


def pred_meta_mlp(snapshot_states, Model, n_input, X_meta, sc):
    """Average predictions across top-K snapshots (TUNE 4)."""
    X_s = torch.from_numpy(sc.transform(X_meta).astype("float32"))
    preds = []
    for state in snapshot_states:
        model = Model(n_input).to(DEVICE)
        model.load_state_dict(state)
        model.eval()
        out = []
        with torch.no_grad():
            for (xb,) in DataLoader(TensorDataset(X_s), batch_size=512):
                out.append(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy())
        preds.append(np.concatenate(out))
    return np.mean(preds, axis=0)


# ============================================================================
# OOF ENGINE
# ============================================================================
def run_oof(X_tr, y_tr, X_val, y_val, X_te, xgb_params, lgb_params):
    NM     = len(ACTIVE_BASE_NAMES)
    oof_p  = np.zeros((len(y_tr), NM), dtype="float32")
    val_p  = np.zeros((len(y_val), NM), dtype="float32")
    te_p   = np.zeros((len(X_te),  NM), dtype="float32")
    oof_pr_per_fold = {n: [] for n in ACTIVE_BASE_NAMES}

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    for fold, (fold_tr_idx, fold_va_idx) in enumerate(skf.split(X_tr, y_tr)):
        Xf_tr, yf_tr = X_tr[fold_tr_idx], y_tr[fold_tr_idx]
        Xf_va, yf_va = X_tr[fold_va_idx],  y_tr[fold_va_idx]

        # 3-way split of fold-train
        rest_frac = OOF_ES_FRAC + OOF_CAL_FRAC
        sss1 = StratifiedShuffleSplit(
            n_splits=1, test_size=rest_frac, random_state=SEED + fold,
        )
        fit_rel, rest_rel = next(sss1.split(Xf_tr, yf_tr))
        Xfit, yfit = Xf_tr[fit_rel],  yf_tr[fit_rel]
        Xrest,yrest= Xf_tr[rest_rel], yf_tr[rest_rel]

        cal_share = OOF_CAL_FRAC / rest_frac
        sss2 = StratifiedShuffleSplit(
            n_splits=1, test_size=cal_share, random_state=SEED + fold + 100,
        )
        es_rel, cal_rel = next(sss2.split(Xrest, yrest))
        Xes,  yes  = Xrest[es_rel],  yrest[es_rel]
        Xcal, ycal = Xrest[cal_rel], yrest[cal_rel]

        n_fit, n_es, n_cal = len(yfit), len(yes), len(ycal)
        print(f"      Fold {fold+1}/{N_FOLDS}  "
              f"[fit={n_fit} | ES={n_es} | cal={n_cal} | OOF={len(yf_va)}]",
              end=" … ", flush=True)

        col = 0

        # XGBoost
        m    = train_xgb_tuned(Xfit, yfit, Xes, yes, xgb_params)
        p_c  = m.predict_proba(Xcal)[:, 1]
        oof_p[fold_va_idx, col] = isotonic_calibrate(p_c, ycal, m.predict_proba(Xf_va)[:, 1])
        val_p[:, col]  += isotonic_calibrate(p_c, ycal, m.predict_proba(X_val)[:, 1]) / N_FOLDS
        te_p[:,  col]  += isotonic_calibrate(p_c, ycal, m.predict_proba(X_te)[:, 1])  / N_FOLDS
        oof_pr_per_fold["XGBoost"].append(safe_pr(yf_va, oof_p[fold_va_idx, col]))
        col += 1

        if HAS_LGB:
            m   = train_lgbm_tuned(Xfit, yfit, Xes, yes, lgb_params)
            p_c = m.predict_proba(Xcal)[:, 1]
            oof_p[fold_va_idx, col] = isotonic_calibrate(p_c, ycal, m.predict_proba(Xf_va)[:, 1])
            val_p[:, col]  += isotonic_calibrate(p_c, ycal, m.predict_proba(X_val)[:, 1]) / N_FOLDS
            te_p[:,  col]  += isotonic_calibrate(p_c, ycal, m.predict_proba(X_te)[:, 1])  / N_FOLDS
            oof_pr_per_fold["LightGBM"].append(safe_pr(yf_va, oof_p[fold_va_idx, col]))
            col += 1

        m   = train_rf(Xfit, yfit)
        p_c = m.predict_proba(Xcal)[:, 1]
        oof_p[fold_va_idx, col] = isotonic_calibrate(p_c, ycal, m.predict_proba(Xf_va)[:, 1])
        val_p[:, col]  += isotonic_calibrate(p_c, ycal, m.predict_proba(X_val)[:, 1]) / N_FOLDS
        te_p[:,  col]  += isotonic_calibrate(p_c, ycal, m.predict_proba(X_te)[:, 1])  / N_FOLDS
        oof_pr_per_fold["RF"].append(safe_pr(yf_va, oof_p[fold_va_idx, col]))
        col += 1

        m, fold_sc = train_logreg(Xfit, yfit)
        p_c = m.predict_proba(fold_sc.transform(Xcal))[:, 1]
        oof_p[fold_va_idx, col] = isotonic_calibrate(
            p_c, ycal, m.predict_proba(fold_sc.transform(Xf_va))[:, 1])
        val_p[:, col]  += isotonic_calibrate(
            p_c, ycal, m.predict_proba(fold_sc.transform(X_val))[:, 1]) / N_FOLDS
        te_p[:,  col]  += isotonic_calibrate(
            p_c, ycal, m.predict_proba(fold_sc.transform(X_te))[:, 1])  / N_FOLDS
        oof_pr_per_fold["LogReg"].append(safe_pr(yf_va, oof_p[fold_va_idx, col]))
        col += 1

        if HAS_TABNET:
            m   = train_tabnet(Xfit, yfit, Xes, yes)
            p_c = m.predict_proba(Xcal)[:, 1]
            oof_p[fold_va_idx, col] = isotonic_calibrate(p_c, ycal, m.predict_proba(Xf_va)[:, 1])
            val_p[:, col]  += isotonic_calibrate(p_c, ycal, m.predict_proba(X_val)[:, 1]) / N_FOLDS
            te_p[:,  col]  += isotonic_calibrate(p_c, ycal, m.predict_proba(X_te)[:, 1])  / N_FOLDS
            oof_pr_per_fold["TabNet"].append(safe_pr(yf_va, oof_p[fold_va_idx, col]))
            col += 1

        print("done")

    # Diagnostics
    print(f"\n      OOF PR-AUC per fold (variance diagnostic):")
    for n in ACTIVE_BASE_NAMES:
        folds_pr = oof_pr_per_fold[n]
        if folds_pr:
            print(f"        {n:<12}  folds={[f'{v:.3f}' for v in folds_pr]}  "
                  f"mean={np.mean(folds_pr):.3f}  std={np.std(folds_pr):.3f}")

    print(f"\n      OOF inter-model correlation (target diversity < 0.85):")
    corr = np.corrcoef(oof_p.T)
    hdr  = "  ".join([f"{b[:5]:>6}" for b in ACTIVE_BASE_NAMES])
    print(f"      {'':6}  {hdr}")
    for i, b in enumerate(ACTIVE_BASE_NAMES):
        row = "  ".join([f"{corr[i,j]:>6.2f}" for j in range(NM)])
        print(f"      {b[:6]:6}  {row}")

    return oof_p, val_p, te_p, oof_pr_per_fold


# ============================================================================
# MAIN LOOP
# ============================================================================
all_rows         = []
all_predictions  = {}
decision_log     = {}
base_rows        = {n: [] for n in ACTIVE_BASE_NAMES}
optuna_log       = {}
gating_log       = {}

HDR = (f"{'Antibiotic':<12} {'AUC':>6} {'PR-AUC':>7} {'F1':>6} {'F2':>6} "
       f"{'MCC':>6} {'Rec':>6} {'Prec':>6} {'Sens90':>7} {'ECE':>6}")

for ab in label_names:
    print(f"\n{'─'*80}")
    print(f"  ANTIBIOTIC : {ab}")
    print(f"{'─'*80}")

    X_tr, y_tr, X_val, y_val, X_te, y_te = get_data(ab)
    print(f"  Train={len(y_tr)} (pos={int(y_tr.sum())}, {100*y_tr.mean():.1f}%)  "
          f"Val={len(y_val)} (pos={int(y_val.sum())})  "
          f"Test={len(y_te)} (pos={int(y_te.sum())})")

    # Val split
    sss_val = StratifiedShuffleSplit(
        n_splits=1, test_size=(1 - VAL_META_FRAC), random_state=SEED,
    )
    meta_va_idx, holdout_va_idx = next(sss_val.split(X_val, y_val))
    X_val_meta, y_val_meta = X_val[meta_va_idx],    y_val[meta_va_idx]
    X_val_hold, y_val_hold = X_val[holdout_va_idx], y_val[holdout_va_idx]
    print(f"  Val split → meta-ES={len(y_val_meta)}  threshold-holdout={len(y_val_hold)}")

    # Optuna (train-side inner split)
    sss_inner = StratifiedShuffleSplit(
        n_splits=1, test_size=OPTUNA_INNER_VAL_FRAC, random_state=SEED,
    )
    inner_tr_idx, inner_va_idx = next(sss_inner.split(X_tr, y_tr))
    X_inner_tr, y_inner_tr = X_tr[inner_tr_idx], y_tr[inner_tr_idx]
    X_inner_va, y_inner_va = X_tr[inner_va_idx], y_tr[inner_va_idx]

    print(f"\n  [Optuna] XGB ({OPTUNA_TRIALS} trials)…", end=" ", flush=True)
    xgb_params, xgb_best_pr = optuna_tune_xgb(
        X_inner_tr, y_inner_tr, X_inner_va, y_inner_va,
    )
    print(f"inner PR-AUC={xgb_best_pr:.4f}")

    lgb_params, lgb_best_pr = (None, 0.0)
    if HAS_LGB:
        print(f"  [Optuna] LGB ({OPTUNA_TRIALS} trials)…", end=" ", flush=True)
        lgb_params, lgb_best_pr = optuna_tune_lgbm(
            X_inner_tr, y_inner_tr, X_inner_va, y_inner_va,
        )
        print(f"inner PR-AUC={lgb_best_pr:.4f}")
    optuna_log[ab] = {
        "xgb_inner_pr": round(xgb_best_pr, 4),
        "xgb_params":   xgb_params,
        "lgb_inner_pr": round(lgb_best_pr, 4),
        "lgb_params":   lgb_params,
    }

    # OOF base models
    print(f"\n  [L0] OOF base models — 3-way fold split …")
    oof_p, val_p_full, te_p, oof_pr_per_fold = run_oof(
        X_tr, y_tr, X_val, y_val, X_te, xgb_params, lgb_params)

    # ============================================================
    # TUNE 1 — Adaptive base gating (per antibiotic, computed from
    # OOF and val ONLY — never test)
    # ============================================================
    print(f"\n      [GATING] Adaptive base learner inclusion check:")
    base_oof_means = []
    base_val_prs   = []
    for i, bn in enumerate(ACTIVE_BASE_NAMES):
        oof_mean = float(np.mean(oof_pr_per_fold[bn])) if oof_pr_per_fold[bn] else 0.0
        val_pr   = safe_pr(y_val, val_p_full[:, i])
        base_oof_means.append(oof_mean)
        base_val_prs.append(val_pr)

    # Decide gating per learner
    active_mask = np.ones(len(ACTIVE_BASE_NAMES), dtype=bool)
    gate_decisions = []
    VAL_PR_GATE_THR = 0.65  # loop এর বাইরে রাখো
    
    for i, bn in enumerate(ACTIVE_BASE_NAMES):
        oof_m = base_oof_means[i]
        val_pr = base_val_prs[i]   # আগে define করো
        gap = abs(val_pr - oof_m)
        reason = []
    
        # condition 1: gap বড় AND val-এও খারাপ
        if gap > GATE_GAP_THR and val_pr < VAL_PR_GATE_THR:
            reason.append(f"gap={gap:.3f}>{GATE_GAP_THR} AND val={val_pr:.3f}<{VAL_PR_GATE_THR}")
    
        # condition 2: OOF PR-AUC খুবই কম
        if oof_m < GATE_PR_THR:
            reason.append(f"OOF_PR={oof_m:.3f}<{GATE_PR_THR}")
    
        if reason:
            active_mask[i] = False
            gate_decisions.append((bn, "GATE_OUT", " AND ".join(reason)))
        else:
            gate_decisions.append((bn, "KEEP", f"OOF={oof_m:.3f} val={val_pr:.3f} gap={gap:.3f}"))
        
    # Safety: never drop below MIN_BASE_AFTER_GATE
    if active_mask.sum() < MIN_BASE_AFTER_GATE:
        # Re-enable the highest-OOF gated learners until threshold met
        order = np.argsort(-np.array(base_oof_means))
        for i in order:
            if not active_mask[i]:
                active_mask[i] = True
                gate_decisions[i] = (ACTIVE_BASE_NAMES[i], "RE_ENABLED",
                                     f"safety: must keep ≥{MIN_BASE_AFTER_GATE} base learners")
            if active_mask.sum() >= MIN_BASE_AFTER_GATE:
                break

    for bn, decision, reason in gate_decisions:
        print(f"        {bn:<12} [{decision:<11}]  {reason}")
    n_active = int(active_mask.sum())
    print(f"      Active base models in meta: {n_active}/{len(ACTIVE_BASE_NAMES)}")

    gating_log[ab] = [{"name": n, "decision": d, "reason": r}
                       for (n, d, r) in gate_decisions]

    # Build meta features with gating
    meta_tr  = make_meta_features(oof_p, active_mask)
    meta_val = make_meta_features(val_p_full, active_mask)
    meta_te  = make_meta_features(te_p, active_mask)
    print(f"      Meta feature dim (after gating): {meta_tr.shape[1]}")

    meta_val_meta = meta_val[meta_va_idx]
    meta_val_hold = meta_val[holdout_va_idx]
    val_base_meta = val_p_full[meta_va_idx]
    val_base_hold = val_p_full[holdout_va_idx]

    # Base reference numbers (full val — for thesis comparison)
    base_pr_full = [safe_pr(y_val, val_p_full[:, i])
                    for i in range(len(ACTIVE_BASE_NAMES))]
    print(f"\n  Base val PR-AUC (full val, reference):")
    for n, v in zip(ACTIVE_BASE_NAMES, base_pr_full):
        print(f"      {n:<10} {v:.4f}")
    for i, bn in enumerate(ACTIVE_BASE_NAMES):
        thr_b, _ = best_thr_mcc(y_val, val_p_full[:, i])
        m_b   = compute_metrics(y_te, te_p[:, i], thr_b)
        base_rows[bn].append({**m_b, "antibiotic": ab})

    # Meta MLP (TUNE 2 + TUNE 4)
    print(f"\n  [L1] Meta MLP (size-aware, top-{META_SNAPSHOTS_K} snapshot avg) …")
    snap_states, meta_sc, meta_best_pr, capacity_tag, ModelCls, n_in = train_meta_mlp(
        meta_tr, y_tr, meta_val_meta, y_val_meta, n_train_samples=len(y_tr))
    print(f"      capacity={capacity_tag}  best_val_pr={meta_best_pr:.4f}  "
          f"snapshots_kept={len(snap_states)}")

    p_meta_val_meta = pred_meta_mlp(snap_states, ModelCls, n_in, meta_val_meta, meta_sc)
    p_meta_val_hold = pred_meta_mlp(snap_states, ModelCls, n_in, meta_val_hold, meta_sc)
    p_meta_val_full = pred_meta_mlp(snap_states, ModelCls, n_in, meta_val,      meta_sc)
    p_meta_te       = pred_meta_mlp(snap_states, ModelCls, n_in, meta_te,       meta_sc)

    best_base_pr_meta = max(safe_pr(y_val_meta, val_base_meta[:, i])
                             for i in range(len(ACTIVE_BASE_NAMES)))
    meta_pr_on_meta   = safe_pr(y_val_meta, p_meta_val_meta)
    delta_meta        = meta_pr_on_meta - best_base_pr_meta
    print(f"      val_meta PR-AUC: meta={meta_pr_on_meta:.4f}  "
          f"best_base={best_base_pr_meta:.4f}  Δ={delta_meta:+.4f}")
    w = np.array(base_pr_full)
    w = w / (w.sum() + 1e-12)
    # Fallback if meta < best_base on val_meta
    p_fallback_val_meta = (val_base_meta * w).sum(axis=1)
    fallback_pr_on_meta = safe_pr(y_val_meta, p_fallback_val_meta)
    use_meta = meta_pr_on_meta >= fallback_pr_on_meta
    if not use_meta:
        p_meta_te       = (te_p * w).sum(axis=1)
        p_meta_val_hold = (val_base_hold * w).sum(axis=1)
        p_meta_val_full = (val_p_full * w).sum(axis=1)
        fallback_pr     = safe_pr(y_val, p_meta_val_full)
        print(f"      ⚠ Fallback: weighted-avg → val PR-AUC={fallback_pr:.4f}")

    decision_log[ab] = {
        "used":                "meta_mlp" if use_meta else "weighted_avg",
        "delta_on_val_meta":   round(delta_meta, 4),
        "meta_capacity":       capacity_tag,
        "n_active_bases":      n_active,
        "n_snapshots_avg":     len(snap_states),
    }

    # ============================================================
    # TUNE 3 — Dual-track threshold (MCC vs F2) on val_holdout ONLY
    # Pick whichever scores higher on geometric mean of MCC and F2
    # ============================================================
    thr_mcc, mcc_at_mcc = best_thr_mcc(y_val_hold, p_meta_val_hold)
    thr_f2,  f2_at_f2   = best_thr_f2(y_val_hold,  p_meta_val_hold)

    pred_at_mcc = (p_meta_val_hold >= thr_mcc).astype(int)
    pred_at_f2  = (p_meta_val_hold >= thr_f2 ).astype(int)
    f2_at_mcc  = float(fbeta_score(y_val_hold, pred_at_mcc, beta=2, zero_division=0))
    mcc_at_f2  = float(matthews_corrcoef(y_val_hold, pred_at_f2))

    # Score = sqrt(MCC * F2) at each threshold (clipped to >=0)
    score_mcc = float(np.sqrt(max(mcc_at_mcc, 0.0) * max(f2_at_mcc, 0.0)))
    score_f2  = float(np.sqrt(max(mcc_at_f2,  0.0) * max(f2_at_f2,  0.0)))

    if score_mcc >= score_f2:
        thr, thr_choice = thr_mcc, "MCC"
    else:
        thr, thr_choice = thr_f2, "F2"
    print(f"\n      Threshold dual-track on val_holdout:")
    print(f"        MCC-thr={thr_mcc:.3f}  → MCC={mcc_at_mcc:.3f} F2={f2_at_mcc:.3f} "
          f"score={score_mcc:.3f}")
    print(f"        F2 -thr={thr_f2:.3f}  → MCC={mcc_at_f2:.3f}  F2={f2_at_f2:.3f}  "
          f"score={score_f2:.3f}")
    print(f"        Selected: [{thr_choice}]  thr={thr:.3f}")

    # Test
    m_out    = compute_metrics(y_te, p_meta_te, thr)
    ci_prauc = bootstrap_ci(y_te, p_meta_te, average_precision_score)
    ci_mcc   = bootstrap_ci(
        y_te, p_meta_te,
        lambda y, p: matthews_corrcoef(y, (p >= thr).astype(int)),
    )
    m_out.update({
        "prauc_ci_lo": ci_prauc[0], "prauc_ci_hi": ci_prauc[1],
        "mcc_ci_lo":   ci_mcc[0],   "mcc_ci_hi":   ci_mcc[1],
        "thr_choice":  thr_choice,
    })

    ece = np.nan
    try:
        prob_true, prob_pred = calibration_curve(
            y_te, np.clip(p_meta_te, 1e-7, 1-1e-7),
            n_bins=10, strategy="uniform",
        )
        bc  = np.histogram(p_meta_te, bins=10, range=(0,1))[0]
        ece = float(np.sum(
            np.abs(prob_true - prob_pred) * bc[:len(prob_true)] / len(y_te)))
    except Exception:
        pass
    m_out["ece"] = ece

    best_te_pr  = max(safe_pr(y_te, te_p[:, i]) for i in range(len(ACTIVE_BASE_NAMES)))
    best_te_mcc = max(matthews_corrcoef(
        y_te, (te_p[:, i] >= best_thr_mcc(y_val, val_p_full[:, i])[0]).astype(int))
        for i in range(len(ACTIVE_BASE_NAMES)))

    d_pr  = m_out["prauc"] - best_te_pr
    d_mcc = m_out["mcc"]   - best_te_mcc

    print(f"\n  ── Result ──────────────────────────────────────────────────────")
    print(f"  PR-AUC : {m_out['prauc']:.4f}  CI=[{ci_prauc[0]:.4f},{ci_prauc[1]:.4f}]  "
          f"Δ_best={d_pr:+.4f}")
    print(f"  MCC    : {m_out['mcc']:.4f}  CI=[{ci_mcc[0]:.4f},{ci_mcc[1]:.4f}]  "
          f"Δ_best={d_mcc:+.4f}")
    print(f"  F1={m_out['f1']:.4f}  F2={m_out['f2']:.4f}  "
          f"Rec={m_out['rec']:.4f}  Prec={m_out['prec']:.4f}  "
          f"Sens@90spec={m_out['sens90']:.4f}  ECE={ece:.4f}")

    all_predictions[ab] = {"y_te": y_te, "p_te": p_meta_te, "thr": thr}
    all_rows.append({"antibiotic": ab,
                     **{f"test_{k}": v for k, v in m_out.items()}})


# ============================================================================
# FINAL SUMMARY
# ============================================================================
df    = pd.DataFrame(all_rows)
mcols = ["test_auc", "test_prauc", "test_f1", "test_f2",
         "test_mcc", "test_brier", "test_rec", "test_prec",
         "test_sens90", "test_acc", "test_ece"]
avg   = df[[c for c in mcols if c in df.columns]].mean(numeric_only=True)

print(f"\n\n{'═'*80}")
print("  FINAL SUMMARY — AMR-CascadeStack v6.1 Tuned Edition")
print(f"  Dataset: {DATASET_PATH.name}  |  Input dim: {INPUT_DIM}  |  "
      f"Base models: {N_BASE_MODELS}")
print(f"{'═'*80}")
print(f"  {HDR}")
print(f"  {'-'*82}")

def ff(v):
    return f"{v:.3f}" if isinstance(v, float) and not np.isnan(v) else "  nan"

for _, r in df.iterrows():
    ece_v = r.get("test_ece", float("nan"))
    print(f"  {r['antibiotic']:<12} {ff(r['test_auc']):>6} {ff(r['test_prauc']):>7} "
          f"{ff(r['test_f1']):>6} {ff(r['test_f2']):>6} {ff(r['test_mcc']):>6} "
          f"{ff(r['test_rec']):>6} {ff(r['test_prec']):>6} "
          f"{ff(r.get('test_sens90', float('nan'))):>7} {ff(ece_v):>6}")

print(f"\n  {'AVERAGE':<12} {ff(avg.get('test_auc', float('nan'))):>6} "
      f"{ff(avg.get('test_prauc', float('nan'))):>7} "
      f"{ff(avg.get('test_f1', float('nan'))):>6} "
      f"{ff(avg.get('test_f2', float('nan'))):>6} "
      f"{ff(avg.get('test_mcc', float('nan'))):>6} "
      f"{ff(avg.get('test_rec', float('nan'))):>6} "
      f"{ff(avg.get('test_prec', float('nan'))):>6} "
      f"{ff(avg.get('test_sens90', float('nan'))):>7} "
      f"{ff(avg.get('test_ece', float('nan'))):>6}")

print(f"\n  Base model averages (test, MCC-thr on full val):")
for bn in ACTIVE_BASE_NAMES:
    rows = base_rows[bn]
    if not rows:
        continue
    print(f"    {bn:<10}  PR-AUC={np.mean([r['prauc'] for r in rows]):.4f}  "
          f"MCC={np.mean([r['mcc'] for r in rows]):.4f}  "
          f"F2={np.mean([r['f2'] for r in rows]):.4f}  "
          f"Sens90={np.nanmean([r['sens90'] for r in rows]):.4f}")

print(f"\n  Per-antibiotic decision (meta vs fallback / capacity / gating):")
for ab, d in decision_log.items():
    flag = "META" if d["used"] == "meta_mlp" else "FALLBACK"
    print(f"    {ab:<10}  [{flag:<8}]  Δ={d['delta_on_val_meta']:+.4f}  "
          f"cap={d['meta_capacity']:<5}  bases_active={d['n_active_bases']}/"
          f"{N_BASE_MODELS}  snapshots={d['n_snapshots_avg']}")

print(f"\n  Threshold choice (per antibiotic):")
for _, r in df.iterrows():
    tc = r.get("test_thr_choice", "?")
    print(f"    {r['antibiotic']:<10}  [{tc}]  thr={r['test_thr']:.3f}  "
          f"F2={r['test_f2']:.3f}  MCC={r['test_mcc']:.3f}")


# ============================================================================
# WILCOXON
# ============================================================================
cas_pr  = [r["test_prauc"] for _, r in df.iterrows()]
cas_mcc = [r["test_mcc"]   for _, r in df.iterrows()]
best_pr, best_mcc = [], []

for ab in label_names:
    pr_list, mcc_list = [], []
    for bn in ACTIVE_BASE_NAMES:
        for r in base_rows[bn]:
            if r["antibiotic"] == ab:
                pr_list.append(r["prauc"]); mcc_list.append(r["mcc"])
    best_pr.append(max(pr_list) if pr_list else 0.0)
    best_mcc.append(max(mcc_list) if mcc_list else 0.0)

print(f"\n  STATISTICAL — v6.1 vs Best Base Model")
print(f"  {'Antibiotic':<12} {'Cas PR':>7} {'BasePR':>7} {'ΔPR':>7} "
      f"{'Cas MCC':>8} {'BaseMCC':>8} {'ΔMCC':>7}")
for i, ab in enumerate(label_names):
    dpr  = cas_pr[i] - best_pr[i]
    dmcc = cas_mcc[i] - best_mcc[i]
    print(f"  {ab:<12} {cas_pr[i]:>7.4f} {best_pr[i]:>7.4f} {dpr:>+7.4f}"
          f"{'✅' if dpr>=0 else '▼'}  "
          f"{cas_mcc[i]:>8.4f} {best_mcc[i]:>8.4f} {dmcc:>+7.4f}"
          f"{'✅' if dmcc>=0 else '▼'}")

for name, cas, bas in [("PR-AUC", cas_pr, best_pr),
                        ("MCC",   cas_mcc, best_mcc)]:
    if len(cas) >= 5:
        try:
            _, p = wilcoxon(cas, bas)
            sig  = "✅ p<0.05" if p < 0.05 else "✗ p≥0.05"
            print(f"  Wilcoxon {name:6s}: p={p:.4f}  [{sig}]")
        except Exception:
            print(f"  Wilcoxon {name}: not computable")


# ============================================================================
# CALIBRATION
# ============================================================================
print(f"\n  CALIBRATION  {'Antibiotic':<10} {'Brier':>7} {'ECE':>7} {'Sens90':>8}")
n_ab   = len(label_names)
n_cols = min(4, n_ab)
n_rows = (n_ab + n_cols - 1) // n_cols
fig_c, axes_c = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows))
axes_c = np.array(axes_c).flatten()

for idx_ab, ab in enumerate(label_names):
    ax    = axes_c[idx_ab]
    y_te  = all_predictions[ab]["y_te"]
    p_te  = all_predictions[ab]["p_te"]
    thr   = all_predictions[ab]["thr"]
    brier = brier_score_loss(y_te, p_te)
    ece   = np.nan
    try:
        prob_true, prob_pred = calibration_curve(
            y_te, np.clip(p_te, 1e-7, 1-1e-7),
            n_bins=10, strategy="uniform",
        )
        ax.plot(prob_pred, prob_true, "o-", color="#3498db",
                label=f"Brier={brier:.3f}")
        ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Perfect")
        bc = np.histogram(p_te, bins=10, range=(0, 1))[0]
        ece = float(np.sum(np.abs(prob_true - prob_pred) *
                            bc[:len(prob_true)] / len(p_te)))
    except Exception:
        ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.axvline(thr, color="red", ls=":", lw=1.5, alpha=0.7,
               label=f"thr={thr:.2f}")
    ax.set_title(f"{ab}\nBrier={brier:.3f}  ECE={ece:.3f}",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted prob"); ax.set_ylabel("Observed freq")
    ax.legend(fontsize=8); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.grid(alpha=0.3)
    sens90 = sensitivity_at_specificity(y_te, p_te, 0.90)
    print(f"      {ab:<10} {brier:>7.4f} {ece:>7.4f} {sens90:>8.4f}")

for ax in axes_c[n_ab:]:
    ax.axis("off")
fig_c.suptitle("Calibration Curves — AMR-CascadeStack v6.1",
               fontsize=13, fontweight="bold")
plt.tight_layout()
cal_path = OUTPUT_DIR / "v6_1_calibration_curves.png"
plt.savefig(cal_path, dpi=200, bbox_inches="tight")
plt.close()


# ============================================================================
# SUMMARY CHART
# ============================================================================
metrics_to_plot = [("test_prauc", "PR-AUC"), ("test_mcc", "MCC"),
                    ("test_f2", "F2"), ("test_sens90", "Sens@90Spec")]
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(6*len(metrics_to_plot), 6))
fig.suptitle("AMR-CascadeStack v6.1 — Test Set Performance",
             fontsize=12, fontweight="bold")
colors = ["#3498db","#2ecc71","#e74c3c","#f39c12","#9b59b6","#1abc9c","#e67e22"]
for ax, (col, lbl) in zip(axes, metrics_to_plot):
    if col not in df.columns: ax.axis("off"); continue
    vals = df[col].values
    bars = ax.bar(df["antibiotic"], vals, color=colors[:len(vals)],
                  edgecolor="black", linewidth=0.5)
    ax.axhline(np.nanmean(vals), color="black", ls="--", lw=1.2,
               label=f"avg={np.nanmean(vals):.3f}")
    ax.set_title(lbl, fontweight="bold"); ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=30)
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{v:.3f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
chart_path = OUTPUT_DIR / "v6_1_summary_chart.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.close()


# ============================================================================
# SAVE
# ============================================================================
# Save block এ
df.to_csv(OUTPUT_DIR / "v6_1_arcadia_v2_results.csv", index=False)
with open(OUTPUT_DIR / "v6_1_optuna_params.json", "w") as f:
    json.dump(optuna_log, f, indent=2)
with open(OUTPUT_DIR / "v6_1_decision_log.json", "w") as f:
    json.dump(decision_log, f, indent=2)
with open(OUTPUT_DIR / "v6_1_gating_log.json", "w") as f:
    json.dump(gating_log, f, indent=2)

print(f"\n  Outputs → {OUTPUT_DIR}")
print(f"    • v6_1_results.csv")
print(f"    • v6_1_optuna_params.json (thesis appendix)")
print(f"    • v6_1_decision_log.json")
print(f"    • v6_1_gating_log.json")
print(f"    • v6_1_calibration_curves.png")
print(f"    • v6_1_summary_chart.png")

print("\n" + "=" * 80)
print("  AMR-CascadeStack v6.1 — COMPLETE")
print("=" * 80)
print(f"""
  5 SAFE TUNINGS APPLIED (vs v6):
    ✅ Adaptive base-learner gating in meta-input
        - Per antibiotic, drop a base learner if OOF→val gap > {GATE_GAP_THR}
          OR OOF PR-AUC < {GATE_PR_THR}
        - Decision uses train OOF + val ONLY — never test
        - Safety floor: never drop below {MIN_BASE_AFTER_GATE} base learners
    ✅ Train-size-aware meta MLP capacity
        - Train < {SMALL_TRAIN_THR} → smaller MLP (32→16) to reduce overfit
        - Train ≥ {SMALL_TRAIN_THR} → original capacity (64→32→16)
    ✅ Dual-track threshold (MCC vs F2) on val_holdout ONLY
        - Pick whichever scores higher on geometric mean of MCC and F2
        - val_holdout used; test never seen for selection
    ✅ Top-{META_SNAPSHOTS_K} snapshot averaging for meta MLP
        - Reduces meta variance at inference
    ✅ Better TabNet config (n_d=24, entmax mask, momentum=0.3, larger batch)

  ALL LEAK-FREE GUARANTEES PRESERVED FROM v6:
    ✅ 3-way fold split (fit / ES / cal) — OOF rows never seen during training
    ✅ Optuna runs on train-side 80/20 inner split — outer val never used
    ✅ Threshold chosen on val_holdout (40% of val) — separate from val_meta
    ✅ Test set never used for any hyperparameter, threshold, or selection
    ✅ All randomness fixed (SEED={SEED})
""")

In [1]:
PA_FAM = "/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/Ecoli_AMR_GenotypePhenotype/Ecoli_AMR_GenotypePhenotype/presence_absence_data/presence_absence_all.fam"
PA_BIM = PA_FAM.replace(".fam", ".bim")

# Sample list (.fam = 6 cols whitespace: FID IID PID MID Sex Pheno)
with open(PA_FAM) as f:
    fam = [l.strip().split() for l in f if l.strip()]
print(f"FAM lines: {len(fam)}")
print("First 3 fam rows:", fam[:3])

# Variant list (.bim = 6 cols: chr varID gen_pos bp_pos allele1 allele2)
with open(PA_BIM) as f:
    bim = [l.strip().split() for l in f if l.strip()]
print(f"\nBIM lines: {len(bim)}")
print("First 5 bim rows:", bim[:5])

# Critical: do .fam IDs match your existing genome_names?
import pickle
with open("/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-b/arcadia_v2_master_features.pkl","rb") as f:
    m = pickle.load(f)
existing_ids = list(m.get("genome_names", []))[:5]
print(f"\nExisting pkl genome_names sample: {existing_ids}")

FAM lines: 7043
First 3 fam rows: [['ERR4037439', 'ERR4037439', '0', '0', '0', '-9'], ['ERR4037517', 'ERR4037517', '0', '0', '0', '-9'], ['ERR4037518', 'ERR4037518', '0', '0', '0', '-9']]

BIM lines: 32441
First 5 bim rows: [['LMHECDEF_00001', '.', '0', '1', '1', '2'], ['LMHECDEF_00002', '.', '0', '1', '1', '2'], ['LMHECDEF_00003', '.', '0', '1', '1', '2'], ['LMHECDEF_00004', '.', '0', '1', '2', '1'], ['LMHECDEF_00005', '.', '0', '1', '2', '1']]

Existing pkl genome_names sample: [np.str_('Escherichia coli strain 123074'), np.str_('Escherichia coli strain 123445'), np.str_('Escherichia coli strain ESBL232B15_13_2E'), np.str_('Escherichia coli strain ESBL370B15_13_2A'), np.str_('Escherichia coli strain ESBL37B15_13_1E')]


In [5]:
"""
build_gene_presence_absence.py
Convert Arcadia PLINK gene-P/A files → aligned (6962, n_pan_genes) binary matrix.

REQUIRED:  pip install pandas-plink
"""
import numpy as np, pandas as pd, pickle
from pathlib import Path
!pip install pandas-plink


# ── PATHS ──────────────────────────────────────────────────────────────────
PA_BASE = "/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/Ecoli_AMR_GenotypePhenotype/Ecoli_AMR_GenotypePhenotype"
PA_BED  = f"{PA_BASE}/presence_absence_data/presence_absence_all.bed"
PA_BIM  = f"{PA_BASE}/presence_absence_data/presence_absence_all.bim"
PA_FAM  = f"{PA_BASE}/presence_absence_data/presence_absence_all.fam"
SRA_MAP = f"{PA_BASE}/phenotype_matrix/SRA_to_genome_name.csv"

# Tomar existing dataset — use whichever has genome_names + sra_ids
MASTER_PKL = "/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-b/arcadia_v2_master_features.pkl"

OUT_PKL = "/kaggle/working/arcadia_v2_gene_presence_absence.pkl"

# ── 1. EXISTING STRAIN ORDER ───────────────────────────────────────────────
with open(MASTER_PKL, "rb") as f:
    M = pickle.load(f)

# Try common keys for sample identifiers
existing_genome_names = list(M.get("genome_names", []))
existing_sra_ids      = list(M.get("sra_ids", []))   # may or may not exist
print(f"Existing samples: {len(existing_genome_names)}")
print(f"  genome_names exemplar: {existing_genome_names[:2]}")
print(f"  sra_ids exemplar: {existing_sra_ids[:2] if existing_sra_ids else 'NOT IN PKL'}")

# ── 2. SRA ↔ Genome.Name MAPPING ────────────────────────────────────────────
sra_map = pd.read_csv(SRA_MAP)
print(f"\nSRA_map cols: {list(sra_map.columns)}")
print(sra_map.head(3))
sra_to_gname = dict(zip(sra_map["SRA.Accession"].astype(str),
                         sra_map["Genome.Name"].astype(str)))
gname_to_sra = {v: k for k, v in sra_to_gname.items()}

# Build: tomar existing → SRA
existing_to_sra = []
for gname in existing_genome_names:
    sra = gname_to_sra.get(str(gname))
    existing_to_sra.append(sra)
matched_to_sra = sum(s is not None for s in existing_to_sra)
print(f"Existing → SRA mapping: {matched_to_sra}/{len(existing_genome_names)}")

# ── 3. PLINK LOAD (memory-efficient via pandas-plink) ──────────────────────
# pip install pandas-plink   (if not installed)
from pandas_plink import read_plink1_bin
G = read_plink1_bin(PA_BED, PA_BIM, PA_FAM, verbose=False)
# G is xarray (sample × variant). values are float (0/1/2 or NaN)
print(f"\nLoaded BED: {G.shape}, dtype={G.dtype}")

fam_sra_ids = [str(s) for s in G.sample.values]   # FAM IDs (column 1)
gene_ids_pa = [str(g) for g in G.variant.values]  # BIM gene IDs
X_full = G.values                                  # (7043, 32441) float

# A1/A2 columns from BIM (need to determine orientation)
# A1/A2 access — robust across pandas_plink versions (diagnostic only)
def _safe_get(da, name):
    try: return [str(x) for x in getattr(da, name).values]
    except (AttributeError, KeyError):
        try: return [str(x) for x in da.coords[name].values]
        except (AttributeError, KeyError): return ["?"] * da.shape[1]
a1 = _safe_get(G, "a1") if hasattr(G, "a1") else _safe_get(G, "allele_0")
a2 = _safe_get(G, "a2") if hasattr(G, "a2") else _safe_get(G, "allele_1")
print(f"First 5 (A1,A2) pairs: {list(zip(a1[:5], a2[:5]))}")

# ── 4. BINARIZATION + ORIENTATION CHECK ────────────────────────────────────
# PLINK 0/1/2 = number of A1 alleles (haploid bacteria = only 0 or 2)
# Convert NaN → 0, then binarize as (value > 0)
# For Arcadia P/A: typically presence = the non-major allele (alt)
# Verify by checking core-gene prevalence (should be ~100% for housekeeping)
X_filled  = np.nan_to_num(X_full, nan=0).astype(np.uint8)
X_bin_raw = (X_filled > 0).astype(np.uint8)   # naive: any non-ref = present

# Prevalence sanity check
gene_prev = X_bin_raw.mean(axis=0)
print("\nGene prevalence distribution (raw orientation):")
print(f"  >0.95 prevalence: {(gene_prev > 0.95).sum()} genes")
print(f"  <0.05 prevalence: {(gene_prev < 0.05).sum()} genes")
print(f"  median prevalence: {np.median(gene_prev):.3f}")

# If median << 0.3 → orientation likely INVERTED (presence is 0, not 2)
INVERT = np.median(gene_prev) < 0.3
if INVERT:
    print("  ⚠️ INVERTED — flipping (presence = value 0, absence = value 2)")
    X_bin = (X_filled == 0).astype(np.uint8)
    gene_prev = X_bin.mean(axis=0)
    print(f"  After flip — median prevalence: {np.median(gene_prev):.3f}")
else:
    X_bin = X_bin_raw
    print("  ✅ Orientation looks correct (median prevalence reasonable)")

# Expect: ~3000-5000 core genes (>95% prev), ~10K-15K accessory (5-95%), 
#         ~15K rare (<5%). E. coli pan-genome standard distribution.

# ── 5. ALIGN TO EXISTING STRAIN ORDER ──────────────────────────────────────
fam_sra_to_idx = {sra: i for i, sra in enumerate(fam_sra_ids)}

n_existing = len(existing_genome_names)
n_pa_genes = len(gene_ids_pa)
X_pa_aligned = np.zeros((n_existing, n_pa_genes), dtype=np.uint8)
matched_mask = np.zeros(n_existing, dtype=bool)

for i, sra in enumerate(existing_to_sra):
    if sra is None: continue
    fam_idx = fam_sra_to_idx.get(sra)
    if fam_idx is None: continue
    X_pa_aligned[i] = X_bin[fam_idx]
    matched_mask[i] = True

print(f"\nAlignment: {matched_mask.sum()}/{n_existing} strains matched to P/A")

# ── 6. OVERLAP WITH SELECTED GENES ─────────────────────────────────────────
# Load gene_filtered.pkl er selected_gene_ids for overlap check
GENE_FILTERED_PKL = "/kaggle/working/arcadia_v2_gene_filtered.pkl"  # adjust path
try:
    with open(GENE_FILTERED_PKL, "rb") as f:
        gf = pickle.load(f)
    selected = set(gf.get("selected_gene_ids", []))
    pa_set = set(gene_ids_pa)
    overlap = selected & pa_set
    print(f"\nOverlap: {len(overlap)}/{len(selected)} selected genes found in P/A catalog")
    print(f"P/A-only genes (not in selected 5000): {len(pa_set - selected)}")
except FileNotFoundError:
    print(f"\nSkip overlap check (gene_filtered.pkl not at {GENE_FILTERED_PKL})")
    overlap = set()

# (just before save) — diagnostic block
print("\n=== RAW VALUE DISTRIBUTION ===")
vals, counts = np.unique(X_filled, return_counts=True)
total = counts.sum()
for v, c in zip(vals.tolist(), counts.tolist()):
    print(f"  value {v}: {c:,} cells ({100*c/total:.2f}%)")

print("\n=== 4 BINARIZATION STRATEGIES ===")
strategies = {
    "S1: X > 0   (naive, current)":   (X_filled > 0).astype(np.uint8),
    "S2: X == 0  (full flip)":        (X_filled == 0).astype(np.uint8),
    "S3: X == 2  (homozygous A1)":    (X_filled == 2).astype(np.uint8),
    "S4: X < 2   (NOT homozygous A1)":(X_filled <  2).astype(np.uint8),
}
for name, Xb in strategies.items():
    prev = Xb.mean(axis=0)
    print(f"\n  {name}:")
    print(f"    median={np.median(prev):.3f}  "
          f"core(>95%)={(prev>0.95).sum():>6,}  "
          f"shell(15-95%)={((prev>0.15)&(prev<=0.95)).sum():>6,}  "
          f"rare(<15%)={(prev<=0.15).sum():>6,}")

# Per-gene encoding hint: do A1 chars vary across genes?
print("\n=== A1 character distribution across all genes ===")
from collections import Counter
print(Counter(a1).most_common())

# ── 7. SAVE ────────────────────────────────────────────────────────────────
out = {
    "X_pa":              X_pa_aligned,        # (n_existing, 32441) uint8
    "gene_ids":          gene_ids_pa,         # 32441 Prokka locus tags
    "sample_genome_names": existing_genome_names,
    "matched_mask":      matched_mask,        # True where P/A available
    "n_samples":         n_existing,
    "n_genes":           n_pa_genes,
    "overlap_with_selected": sorted(list(overlap)),
    "inverted_encoding": bool(INVERT),
    "gene_prevalence_stats": {
        "median": float(np.median(gene_prev)),
        "n_core_gt95":  int((gene_prev > 0.95).sum()),
        "n_rare_lt05":  int((gene_prev < 0.05).sum()),
        "n_accessory":  int(((gene_prev >= 0.05) & (gene_prev <= 0.95)).sum()),
    },
}
with open(OUT_PKL, "wb") as f:
    pickle.dump(out, f)
print(f"\n✅ Saved: {OUT_PKL}")
print(f"  X_pa: {X_pa_aligned.shape}, memory={X_pa_aligned.nbytes/1e6:.1f} MB")
print(f"  Prevalence: median={out['gene_prevalence_stats']['median']:.3f}, "
      f"core={out['gene_prevalence_stats']['n_core_gt95']}, "
      f"accessory={out['gene_prevalence_stats']['n_accessory']}, "
      f"rare={out['gene_prevalence_stats']['n_rare_lt05']}")

Existing samples: 6962
  genome_names exemplar: [np.str_('Escherichia coli strain 123074'), np.str_('Escherichia coli strain 123445')]
  sra_ids exemplar: [np.str_('ERR1197942'), np.str_('ERR1197943')]

SRA_map cols: ['Unnamed: 0', 'Genome.Name', 'SRA.Accession']
   Unnamed: 0                                        Genome.Name SRA.Accession
0      3081.0  Escherichia coli c0fb0992-7bb9-11e9-a8d3-68b59...    ERR4037439
1      3082.0  Escherichia coli ca7bc2ea-7bb9-11e9-a8d3-68b59...    ERR4037517
2      3083.0  Escherichia coli ca96892c-7bb9-11e9-a8d3-68b59...    ERR4037518
Existing → SRA mapping: 6962/6962

Loaded BED: (7043, 32441), dtype=float32
First 5 (A1,A2) pairs: [('2', '?'), ('2', '?'), ('2', '?'), ('1', '?'), ('1', '?')]

Gene prevalence distribution (raw orientation):
  >0.95 prevalence: 17762 genes
  <0.05 prevalence: 0 genes
  median prevalence: 0.966
  ✅ Orientation looks correct (median prevalence reasonable)

Alignment: 6949/6962 strains matched to P/A

Skip overlap chec

In [6]:
# Re-run JUST the binarization + save steps with S2

X_filled = np.nan_to_num(X_full, nan=0).astype(np.uint8)
X_bin = (X_filled == 0).astype(np.uint8)        # S2: presence = value 0
prev = X_bin.mean(axis=0)

# Refined distribution at multiple cutoffs
print("=== CORRECTED PREVALENCE DISTRIBUTION (S2) ===")
print(f"median: {np.median(prev):.4f}")
print(f"per-strain gene count: mean={X_bin.sum(axis=1).mean():.0f}, "
      f"median={int(np.median(X_bin.sum(axis=1)))}, "
      f"range=[{X_bin.sum(axis=1).min()}, {X_bin.sum(axis=1).max()}]")
print(f"\n  Core    (>95%):  {(prev > 0.95).sum():>6,}")
print(f"  Soft    (90-95%): {((prev > 0.90) & (prev <= 0.95)).sum():>6,}")
print(f"  Shell-H (50-90%): {((prev > 0.50) & (prev <= 0.90)).sum():>6,}")
print(f"  Shell-L (15-50%): {((prev > 0.15) & (prev <= 0.50)).sum():>6,}")
print(f"  Cloud   (5-15%):  {((prev > 0.05) & (prev <= 0.15)).sum():>6,}")
print(f"  Singleton (<5%):  {(prev <= 0.05).sum():>6,}")

# Re-align (same as before, but with corrected X_bin)
X_pa_aligned = np.zeros((n_existing, n_pa_genes), dtype=np.uint8)
matched_mask = np.zeros(n_existing, dtype=bool)
fam_sra_to_idx = {sra: i for i, sra in enumerate(fam_sra_ids)}
for i, sra in enumerate(existing_to_sra):
    if sra is None: continue
    fam_idx = fam_sra_to_idx.get(sra)
    if fam_idx is None: continue
    X_pa_aligned[i] = X_bin[fam_idx]
    matched_mask[i] = True

# Overlap with selected gene IDs (point to correct path now)
GENE_FILTERED_PKL = "/kaggle/input/notebooks/faiazabrarzaman/<your-stage-1.5-notebook>/arcadia_v2_gene_filtered.pkl"
# OR if you have it locally:
# GENE_FILTERED_PKL = "/kaggle/working/arcadia_v2_gene_filtered.pkl"

try:
    with open(GENE_FILTERED_PKL, "rb") as f:
        gf = pickle.load(f)
    selected = set(gf.get("selected_gene_ids", []))
    pa_set = set(gene_ids_pa)
    overlap = selected & pa_set
    print(f"\n=== OVERLAP ===")
    print(f"Selected (from gene_filtered.pkl):  {len(selected)}")
    print(f"P/A catalog: {len(pa_set)}")
    print(f"Overlap: {len(overlap)} ({100*len(overlap)/len(selected):.1f}% of selected)")
    print(f"P/A-only (NEW info Expert B sees): {len(pa_set - selected)}")
except Exception as e:
    print(f"\nOverlap skipped: {e}")
    overlap = set()

# Save corrected
out = {
    "X_pa":              X_pa_aligned,
    "gene_ids":          gene_ids_pa,
    "sample_genome_names": existing_genome_names,
    "matched_mask":      matched_mask,
    "n_samples":         n_existing,
    "n_genes":           n_pa_genes,
    "overlap_with_selected": sorted(list(overlap)),
    "encoding_rule":     "present = (PLINK_value == 0); A1=REF, --keep-allele-order",
    "per_strain_gene_count_stats": {
        "mean":   float(X_bin.sum(axis=1).mean()),
        "median": int(np.median(X_bin.sum(axis=1))),
        "min":    int(X_bin.sum(axis=1).min()),
        "max":    int(X_bin.sum(axis=1).max()),
    },
    "gene_prevalence_stats": {
        "median":     float(np.median(prev)),
        "core_gt95":  int((prev > 0.95).sum()),
        "shell":      int(((prev > 0.05) & (prev <= 0.95)).sum()),
        "rare_lt05":  int((prev <= 0.05).sum()),
    },
}
with open(OUT_PKL, "wb") as f:
    pickle.dump(out, f)
print(f"\n✅ Re-saved with S2: {OUT_PKL}")

=== CORRECTED PREVALENCE DISTRIBUTION (S2) ===
median: 0.0342
per-strain gene count: mean=3727, median=3349, range=[2141, 8333]

  Core    (>95%):       0
  Soft    (90-95%):      0
  Shell-H (50-90%):      0
  Shell-L (15-50%): 10,137
  Cloud   (5-15%):   4,542
  Singleton (<5%):  17,762

Overlap skipped: [Errno 2] No such file or directory: '/kaggle/input/notebooks/faiazabrarzaman/<your-stage-1.5-notebook>/arcadia_v2_gene_filtered.pkl'

✅ Re-saved with S2: /kaggle/working/arcadia_v2_gene_presence_absence.pkl


In [8]:
"""
TriLens v10 — FINAL pre-flight check
Verifies every data dependency before architecture build.
"""
import pickle, numpy as np
from pathlib import Path

PA_PKL = "/kaggle/working/arcadia_v2_gene_presence_absence.pkl"
# Tomar gene_filtered.pkl Kaggle e upload korte hobe ekti dataset/notebook hisebe.
# Adjust below path:
GF_PKL = "/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/arcadia_v2_gene_filtered.pkl"
MASTER_PKL = "/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-b/arcadia_v2_master_features.pkl"

ok, warn = [], []
def OK(m):   ok.append(f"✅ {m}");   print(f"✅ {m}")
def WARN(m): warn.append(f"⚠️ {m}"); print(f"⚠️ {m}")
def FAIL(m): print(f"❌ {m}");      raise SystemExit(m)

# ────────────────────────────────────────────────────────────
# 1. P/A pkl integrity
# ────────────────────────────────────────────────────────────
print("\n[1] P/A pkl")
with open(PA_PKL, "rb") as f: pa = pickle.load(f)
print(f"  keys: {list(pa.keys())}")
assert pa["X_pa"].shape == (6962, 32441), f"X_pa shape wrong: {pa['X_pa'].shape}"
OK(f"X_pa shape (6962, 32441), dtype={pa['X_pa'].dtype}")
assert pa["X_pa"].dtype == np.uint8, "X_pa must be uint8"
OK("dtype uint8")
ps_count = pa["X_pa"].sum(axis=1)
assert 3000 < ps_count.mean() < 5000, f"per-strain mean {ps_count.mean()} suspect"
OK(f"per-strain genes: mean={ps_count.mean():.0f}, median={int(np.median(ps_count))}")
assert pa["matched_mask"].sum() >= 6900, f"only {pa['matched_mask'].sum()} matched"
OK(f"matched samples: {pa['matched_mask'].sum()}/6962")
assert "encoding_rule" in pa, "encoding_rule field missing — re-run S2 script"
print(f"  encoding_rule: {pa['encoding_rule']}")
assert "== 0" in pa["encoding_rule"], "encoding NOT S2 — re-run with corrected script"
OK("S2 encoding confirmed (present = PLINK_value==0)")

# ────────────────────────────────────────────────────────────
# 2. gene_filtered.pkl (Expert A + Expert C inputs)
# ────────────────────────────────────────────────────────────
print("\n[2] gene_filtered.pkl")
try:
    with open(GF_PKL, "rb") as f: gf = pickle.load(f)
except FileNotFoundError:
    FAIL(f"gene_filtered.pkl NOT FOUND at {GF_PKL}\n"
         f"   → upload as Kaggle dataset, update GF_PKL path, re-run.")
print(f"  keys: {list(gf.keys())}")
for k in ["X_snp_filtered", "gene_to_range", "selected_gene_ids", "Y", "label_mask", "random_split"]:
    assert k in gf, f"missing key in gene_filtered.pkl: {k}"
OK("all required keys present")
print(f"  X_snp_filtered shape: {gf['X_snp_filtered'].shape}  dtype={gf['X_snp_filtered'].dtype}")
print(f"  selected_gene_ids: {len(gf['selected_gene_ids'])}")
print(f"  gene_to_range entries: {len(gf['gene_to_range'])}")
lens = np.array([e-s for s,e in gf['gene_to_range'].values()])
print(f"  per-gene SNP count: median={int(np.median(lens))}, max={lens.max()}, min={lens.min()}, n_lt3={(lens<3).sum()}")
if (lens < 3).sum() > 0:
    WARN(f"{(lens<3).sum()} genes have <3 SNPs — Expert A CNN fallback needed")
print(f"  Y shape: {gf['Y'].shape}, label_mask shape: {gf['label_mask'].shape}")
assert gf['Y'].shape == (6962, 7), "Y shape wrong"
OK("Y, label_mask shapes correct")

# ────────────────────────────────────────────────────────────
# 3. OVERLAP — THE CONFUSION TO RESOLVE
# ────────────────────────────────────────────────────────────
print("\n[3] Overlap: selected_gene_ids ↔ P/A gene_ids")
sel  = set(str(g) for g in gf["selected_gene_ids"])
pa_g = set(str(g) for g in pa["gene_ids"])
overlap = sel & pa_g
print(f"  selected_gene_ids:  {len(sel)}")
print(f"  P/A catalog genes:  {len(pa_g)}")
print(f"  overlap:            {len(overlap)} ({100*len(overlap)/len(sel):.1f}% of selected)")
print(f"  P/A-only (new info): {len(pa_g - sel)}")
print(f"  selected-only (no P/A): {len(sel - overlap)}")
if len(overlap) < 4000:
    WARN(f"overlap only {len(overlap)} — Experts A & C will share <{len(overlap)} genes with B; some Expert A genes won't have P/A counterpart")
else:
    OK(f"overlap {len(overlap)} ≥ 4000 — Expert A/C/B share substantial gene set")
print(f"  selected_gene_ids sample: {list(sel)[:3]}")
print(f"  P/A gene_ids sample:      {list(pa_g)[:3]}")

# ────────────────────────────────────────────────────────────
# 4. Strain order alignment across pkls
# ────────────────────────────────────────────────────────────
print("\n[4] Strain order alignment")
pa_strains = pa.get("sample_genome_names", [])
gf_strains = gf.get("genome_names", []) or gf.get("selected_genome_names", [])
if not gf_strains:
    with open(MASTER_PKL, "rb") as f: m = pickle.load(f)
    gf_strains = m.get("genome_names", [])
    WARN("gene_filtered.pkl lacks genome_names; using master_features.pkl")
print(f"  pa strains: {len(pa_strains)}, gf strains: {len(gf_strains)}")
mismatch = [(i, str(a), str(b)) for i, (a,b) in enumerate(zip(pa_strains, gf_strains)) if str(a) != str(b)]
if mismatch:
    FAIL(f"strain order MISMATCH at {len(mismatch)} positions! "
         f"first: idx={mismatch[0][0]}, pa='{mismatch[0][1]}', gf='{mismatch[0][2]}'")
OK(f"all {len(pa_strains)} strains aligned identically")

# ────────────────────────────────────────────────────────────
# 5. Splits sanity
# ────────────────────────────────────────────────────────────
print("\n[5] Splits")
rs = gf["random_split"]
ps = gf.get("phylogeny_split")
for name, sp in [("random", rs), ("phylogeny", ps)]:
    if sp is None:
        WARN(f"{name}_split MISSING — Pillar 1 leakage test cannot run")
        continue
    keys = list(sp.keys())
    # accept both 'train'/'val'/'test' and 'train_idx'/'val_idx'/'test_idx'
    tr = sp.get("train") or sp.get("train_idx")
    va = sp.get("val")   or sp.get("val_idx")
    te = sp.get("test")  or sp.get("test_idx")
    n  = len(tr) + len(va) + len(te)
    print(f"  {name}: train={len(tr)} val={len(va)} test={len(te)} total={n}  keys={keys}")
    assert n == 6962, f"{name} split total {n} != 6962"
OK("splits consistent")

# ────────────────────────────────────────────────────────────
# 6. Memory + disk estimates
# ────────────────────────────────────────────────────────────
print("\n[6] Resource estimates")
n_genes_A = len(gf['selected_gene_ids'])
L_max = 64
disk_pergene = 6962 * n_genes_A * L_max * 4 / 1e9
print(f"  Expert A pergene cache (Lmax=64, float32): ~{disk_pergene:.1f} GB on disk")
mem_A_B16 = 16 * n_genes_A * L_max * 32 * 4 / 1e9
print(f"  Expert A activations B=16: ~{mem_A_B16:.2f} GB peak")
mem_B = 16 * 32441 * 32 * 4 / 1e9
print(f"  Expert B activations B=16: ~{mem_B:.2f} GB peak")
print(f"  Total expert peak B=16: ~{mem_A_B16+mem_B+0.5:.1f} GB (target < 11 GB)")

print("\n" + "="*60)
print(f"SUMMARY: {len(ok)} ok, {len(warn)} warnings, 0 fails")
print("="*60)
for w in warn: print(w)
if not warn: print("✅ ZERO confusion — all data verified. Ready for TriLens v10.")


[1] P/A pkl
  keys: ['X_pa', 'gene_ids', 'sample_genome_names', 'matched_mask', 'n_samples', 'n_genes', 'overlap_with_selected', 'encoding_rule', 'per_strain_gene_count_stats', 'gene_prevalence_stats']
✅ X_pa shape (6962, 32441), dtype=uint8
✅ dtype uint8
✅ per-strain genes: mean=3709, median=3331
✅ matched samples: 6949/6962
  encoding_rule: present = (PLINK_value == 0); A1=REF, --keep-allele-order
✅ S2 encoding confirmed (present = PLINK_value==0)

[2] gene_filtered.pkl
  keys: ['X_snp_filtered', 'snp_positions_filtered', 'selected_gene_ids', 'gene_to_range', 'Y', 'label_mask', 'label_names', 'antibiotic_to_index', 'random_split', 'phylogeny_split', 'phylogroup_onehot', 'phylogroup_names', 'genome_names', 'sra_ids', 'n_samples', 'n_genes', 'n_snps_filtered', 'n_snps_original', 'filter_metadata', 'stage', 'version']
✅ all required keys present
  X_snp_filtered shape: (6962, 262225)  dtype=int8
  selected_gene_ids: 5000
  gene_to_range entries: 5000
  per-gene SNP count: median=32, ma

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [9]:
# RE-RUN ON KAGGLE (just the gene-ID extraction + re-save)
import numpy as np, pickle
from pandas_plink import read_plink1_bin

PA_BED = "/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/Ecoli_AMR_GenotypePhenotype/Ecoli_AMR_GenotypePhenotype/presence_absence_data/presence_absence_all.bed"
PA_BIM = PA_BED.replace(".bed", ".bim")
PA_FAM = PA_BED.replace(".bed", ".fam")
OLD_PKL = "/kaggle/working/arcadia_v2_gene_presence_absence.pkl"
NEW_PKL = "/kaggle/working/arcadia_v2_gene_presence_absence_v2.pkl"

G = read_plink1_bin(PA_BED, PA_BIM, PA_FAM, verbose=False)
gene_ids_chrom   = [str(g) for g in G.chrom.values]    # ← CORRECT source
gene_ids_variant = [str(g) for g in G.variant.values]  # ← wrong (just for comparison)

print("chrom column (CORRECT) first 5:", gene_ids_chrom[:5])
print("variant column (auto-gen)  first 5:", gene_ids_variant[:5])
print(f"chrom uniqueness: {len(set(gene_ids_chrom))}/{len(gene_ids_chrom)}")

# Verify: should match BIM file line 1 col 1
with open(PA_BIM) as f: bim0 = f.readline().strip().split()
assert gene_ids_chrom[0] == bim0[0], f"chrom[0]={gene_ids_chrom[0]} != bim[0]={bim0[0]}"
print(f"✅ chrom column matches BIM line 1: {gene_ids_chrom[0]}")

# Load old pkl, swap gene_ids, re-save
with open(OLD_PKL, "rb") as f: old = pickle.load(f)
old["gene_ids"] = gene_ids_chrom
old["gene_ids_source"] = "BIM chromosome column (chrom) — Prokka locus tags"
with open(NEW_PKL, "wb") as f: pickle.dump(old, f)
print(f"✅ Re-saved with corrected gene_ids: {NEW_PKL}")
print(f"  X_pa shape preserved: {old['X_pa'].shape}")
print(f"  gene_ids sample: {old['gene_ids'][:3]} ... {old['gene_ids'][-2:]}")


chrom column (CORRECT) first 5: ['LMHECDEF_00001', 'LMHECDEF_00002', 'LMHECDEF_00003', 'LMHECDEF_00004', 'LMHECDEF_00005']
variant column (auto-gen)  first 5: ['variant0', 'variant1', 'variant2', 'variant3', 'variant4']
chrom uniqueness: 32441/32441
✅ chrom column matches BIM line 1: LMHECDEF_00001
✅ Re-saved with corrected gene_ids: /kaggle/working/arcadia_v2_gene_presence_absence_v2.pkl
  X_pa shape preserved: (6962, 32441)
  gene_ids sample: ['LMHECDEF_00001', 'LMHECDEF_00002', 'LMHECDEF_00003'] ... ['Cluster_13961_+_+_GCA_003335465.1_ASM333546v1_+_+_PNGICKPE_04281_+_+_PNGICKPE_04282_+_+_CO_F', 'Cluster_4488_+_+_GCA_003335465.1_ASM333546v1_+_+_PNGICKPE_04306_+_+_PNGICKPE_04307_+_+_DT']


In [10]:
"""
TriLens v10 — FINAL pre-flight check
Verifies every data dependency before architecture build.
"""
import pickle, numpy as np
from pathlib import Path

PA_PKL = "/kaggle/working/arcadia_v2_gene_presence_absence_v2.pkl"
# Tomar gene_filtered.pkl Kaggle e upload korte hobe ekti dataset/notebook hisebe.
# Adjust below path:
GF_PKL = "/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/arcadia_v2_gene_filtered.pkl"
MASTER_PKL = "/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-b/arcadia_v2_master_features.pkl"

ok, warn = [], []
def OK(m):   ok.append(f"✅ {m}");   print(f"✅ {m}")
def WARN(m): warn.append(f"⚠️ {m}"); print(f"⚠️ {m}")
def FAIL(m): print(f"❌ {m}");      raise SystemExit(m)

# ────────────────────────────────────────────────────────────
# 1. P/A pkl integrity
# ────────────────────────────────────────────────────────────
print("\n[1] P/A pkl")
with open(PA_PKL, "rb") as f: pa = pickle.load(f)
print(f"  keys: {list(pa.keys())}")
assert pa["X_pa"].shape == (6962, 32441), f"X_pa shape wrong: {pa['X_pa'].shape}"
OK(f"X_pa shape (6962, 32441), dtype={pa['X_pa'].dtype}")
assert pa["X_pa"].dtype == np.uint8, "X_pa must be uint8"
OK("dtype uint8")
ps_count = pa["X_pa"].sum(axis=1)
assert 3000 < ps_count.mean() < 5000, f"per-strain mean {ps_count.mean()} suspect"
OK(f"per-strain genes: mean={ps_count.mean():.0f}, median={int(np.median(ps_count))}")
assert pa["matched_mask"].sum() >= 6900, f"only {pa['matched_mask'].sum()} matched"
OK(f"matched samples: {pa['matched_mask'].sum()}/6962")
assert "encoding_rule" in pa, "encoding_rule field missing — re-run S2 script"
print(f"  encoding_rule: {pa['encoding_rule']}")
assert "== 0" in pa["encoding_rule"], "encoding NOT S2 — re-run with corrected script"
OK("S2 encoding confirmed (present = PLINK_value==0)")

# ────────────────────────────────────────────────────────────
# 2. gene_filtered.pkl (Expert A + Expert C inputs)
# ────────────────────────────────────────────────────────────
print("\n[2] gene_filtered.pkl")
try:
    with open(GF_PKL, "rb") as f: gf = pickle.load(f)
except FileNotFoundError:
    FAIL(f"gene_filtered.pkl NOT FOUND at {GF_PKL}\n"
         f"   → upload as Kaggle dataset, update GF_PKL path, re-run.")
print(f"  keys: {list(gf.keys())}")
for k in ["X_snp_filtered", "gene_to_range", "selected_gene_ids", "Y", "label_mask", "random_split"]:
    assert k in gf, f"missing key in gene_filtered.pkl: {k}"
OK("all required keys present")
print(f"  X_snp_filtered shape: {gf['X_snp_filtered'].shape}  dtype={gf['X_snp_filtered'].dtype}")
print(f"  selected_gene_ids: {len(gf['selected_gene_ids'])}")
print(f"  gene_to_range entries: {len(gf['gene_to_range'])}")
lens = np.array([e-s for s,e in gf['gene_to_range'].values()])
print(f"  per-gene SNP count: median={int(np.median(lens))}, max={lens.max()}, min={lens.min()}, n_lt3={(lens<3).sum()}")
if (lens < 3).sum() > 0:
    WARN(f"{(lens<3).sum()} genes have <3 SNPs — Expert A CNN fallback needed")
print(f"  Y shape: {gf['Y'].shape}, label_mask shape: {gf['label_mask'].shape}")
assert gf['Y'].shape == (6962, 7), "Y shape wrong"
OK("Y, label_mask shapes correct")

# ────────────────────────────────────────────────────────────
# 3. OVERLAP — THE CONFUSION TO RESOLVE
# ────────────────────────────────────────────────────────────
print("\n[3] Overlap: selected_gene_ids ↔ P/A gene_ids")
sel  = set(str(g) for g in gf["selected_gene_ids"])
pa_g = set(str(g) for g in pa["gene_ids"])
overlap = sel & pa_g
print(f"  selected_gene_ids:  {len(sel)}")
print(f"  P/A catalog genes:  {len(pa_g)}")
print(f"  overlap:            {len(overlap)} ({100*len(overlap)/len(sel):.1f}% of selected)")
print(f"  P/A-only (new info): {len(pa_g - sel)}")
print(f"  selected-only (no P/A): {len(sel - overlap)}")
if len(overlap) < 4000:
    WARN(f"overlap only {len(overlap)} — Experts A & C will share <{len(overlap)} genes with B; some Expert A genes won't have P/A counterpart")
else:
    OK(f"overlap {len(overlap)} ≥ 4000 — Expert A/C/B share substantial gene set")
print(f"  selected_gene_ids sample: {list(sel)[:3]}")
print(f"  P/A gene_ids sample:      {list(pa_g)[:3]}")

# ────────────────────────────────────────────────────────────
# 4. Strain order alignment across pkls
# ────────────────────────────────────────────────────────────
print("\n[4] Strain order alignment")
pa_strains = pa.get("sample_genome_names", [])
gf_strains_raw = gf.get("genome_names")
if gf_strains_raw is None:
    gf_strains_raw = gf.get("selected_genome_names")
gf_strains = [str(x) for x in (gf_strains_raw if gf_strains_raw is not None else [])]

if not gf_strains:
    with open(MASTER_PKL, "rb") as f: m = pickle.load(f)
    gf_strains = m.get("genome_names", [])
    WARN("gene_filtered.pkl lacks genome_names; using master_features.pkl")
print(f"  pa strains: {len(pa_strains)}, gf strains: {len(gf_strains)}")
mismatch = [(i, str(a), str(b)) for i, (a,b) in enumerate(zip(pa_strains, gf_strains)) if str(a) != str(b)]
if mismatch:
    FAIL(f"strain order MISMATCH at {len(mismatch)} positions! "
         f"first: idx={mismatch[0][0]}, pa='{mismatch[0][1]}', gf='{mismatch[0][2]}'")
OK(f"all {len(pa_strains)} strains aligned identically")

# ────────────────────────────────────────────────────────────
# 5. Splits sanity
# ────────────────────────────────────────────────────────────
print("\n[5] Splits")
rs = gf["random_split"]
ps = gf.get("phylogeny_split")
for name, sp in [("random", rs), ("phylogeny", ps)]:
    if sp is None:
        WARN(f"{name}_split MISSING — Pillar 1 leakage test cannot run")
        continue
    keys = list(sp.keys())
    # accept both 'train'/'val'/'test' and 'train_idx'/'val_idx'/'test_idx'
    tr = sp.get("train") or sp.get("train_idx")
    va = sp.get("val")   or sp.get("val_idx")
    te = sp.get("test")  or sp.get("test_idx")
    n  = len(tr) + len(va) + len(te)
    print(f"  {name}: train={len(tr)} val={len(va)} test={len(te)} total={n}  keys={keys}")
    assert n == 6962, f"{name} split total {n} != 6962"
OK("splits consistent")

# ────────────────────────────────────────────────────────────
# 6. Memory + disk estimates
# ────────────────────────────────────────────────────────────
print("\n[6] Resource estimates")
n_genes_A = len(gf['selected_gene_ids'])
L_max = 64
disk_pergene = 6962 * n_genes_A * L_max * 4 / 1e9
print(f"  Expert A pergene cache (Lmax=64, float32): ~{disk_pergene:.1f} GB on disk")
mem_A_B16 = 16 * n_genes_A * L_max * 32 * 4 / 1e9
print(f"  Expert A activations B=16: ~{mem_A_B16:.2f} GB peak")
mem_B = 16 * 32441 * 32 * 4 / 1e9
print(f"  Expert B activations B=16: ~{mem_B:.2f} GB peak")
print(f"  Total expert peak B=16: ~{mem_A_B16+mem_B+0.5:.1f} GB (target < 11 GB)")

print("\n" + "="*60)
print(f"SUMMARY: {len(ok)} ok, {len(warn)} warnings, 0 fails")
print("="*60)
for w in warn: print(w)
if not warn: print("✅ ZERO confusion — all data verified. Ready for TriLens v10.")


[1] P/A pkl
  keys: ['X_pa', 'gene_ids', 'sample_genome_names', 'matched_mask', 'n_samples', 'n_genes', 'overlap_with_selected', 'encoding_rule', 'per_strain_gene_count_stats', 'gene_prevalence_stats', 'gene_ids_source']
✅ X_pa shape (6962, 32441), dtype=uint8
✅ dtype uint8
✅ per-strain genes: mean=3709, median=3331
✅ matched samples: 6949/6962
  encoding_rule: present = (PLINK_value == 0); A1=REF, --keep-allele-order
✅ S2 encoding confirmed (present = PLINK_value==0)

[2] gene_filtered.pkl
  keys: ['X_snp_filtered', 'snp_positions_filtered', 'selected_gene_ids', 'gene_to_range', 'Y', 'label_mask', 'label_names', 'antibiotic_to_index', 'random_split', 'phylogeny_split', 'phylogroup_onehot', 'phylogroup_names', 'genome_names', 'sra_ids', 'n_samples', 'n_genes', 'n_snps_filtered', 'n_snps_original', 'filter_metadata', 'stage', 'version']
✅ all required keys present
  X_snp_filtered shape: (6962, 262225)  dtype=int8
  selected_gene_ids: 5000
  gene_to_range entries: 5000
  per-gene SNP c